# KLTN attack v3.2 — Dynamic concept attention trên Kaggle T4 ×2

GPU 0 chạy ensemble FOA B16/B32/Laion; GPU 1 chạy Qwen3-VL-8B NF4/FP16.
Vùng salient được tính lại từ **PNG adversarial hiện tại**, mặc định mỗi 30/300 updates.
Map mới áp dụng từ update tiếp theo; chọn phase theo tiến độ tuyệt đối, không quay về phase đầu.
Không dùng map cũ thay thế khi attention lỗi. Giữ global/local OT, dynamic ensemble weights,
hai lượt full/crop của FOA và FGSM/projection epsilon 16 trong miền pixel [0,255].

**Chạy:** Kaggle → Internet On, T4 ×2 → Add Input dataset → sửa `DATASET_ROOT` → Run All trong session mới.
Mặc định `NUM_SAMPLES=1` để smoke test; đặt `None` để chạy hết bảng caption (100 mẫu nhúng),
hoặc cung cấp `CAPTION_INPUT_PATH` JSONL cho tập lớn hơn. Khớp bằng đường dẫn cặp ảnh và checksum PNG
sau preprocessing, không khớp theo thứ tự dòng. Caption bảng ngoài giữ schema của bảng nhúng.

**Tối ưu:** SDPA cho các layer trước layer 29; tại layer 29 chỉ tính các hàng Q/K của concept,
giữ softmax causal trên toàn bộ keys rồi mới lấy image keys. Dừng forward ngay sau khi lấy map,
không chạy các decoder layer phía sau hay LM head. Cache vision encoder trong cùng một checkpoint;
không tái dùng vision features giữa các ảnh/steps. Trả guide map trước, tính các concept phụ trên GPU 1
trong lúc GPU 0 tiếp tục attack. Chỉ một worker Qwen, có backpressure và drain trước khi kết thúc mẫu.
Worker eval/inference không đọc hay khôi phục RNG của attack.

`FOA_AUTOCAST=True` dùng FP16 forward CLIP và FP32 features/OT/K-means; scale objective dương
trước khi lấy sign gradient để giảm underflow. Đây là thay đổi số học, **không cam kết ảnh bit-exact với v3.1**.
Đặt `FOA_AUTOCAST=False` khi ablation FP32. Loss/seed reset theo mẫu (`seed + index`) để resume ổn định;
v3.1 giữ loss history xuyên mẫu nên chuỗi nhiều mẫu có thể khác.

**Artifacts mặc định:** 11 checkpoint PNG (0,30,…,300), guide maps ở mọi checkpoint; source/target/extra
ở 0,150,300 (`OBSERVE_ALL_CONCEPTS_EVERY=5`, đặt 1 để lưu mọi checkpoint).
Trace mọi step, schedule mỗi refresh, checksum, timing, status, manifest và ba PNG tương thích captioning.
Chỉ vẽ timeline cho hai mẫu đầu để giảm overhead; mọi mẫu vẫn có PNG/NPZ để vẽ lại bằng cell inspect/reprobe.
`LIVE_PREVIEW=False` mặc định. ZIP dùng mức nén thấp, không chứa weights.

**Resume:** đặt `RESUME_RUN_DIR=Path('/kaggle/working/.../kltn_attack_v3_2_dynamic_...')` rồi chạy lại.
Notebook kiểm tra cấu hình, bỏ qua mẫu hoàn tất; mẫu lỗi được giữ trong `failed_samples/` rồi thử lại.
Nếu run cũ ở Kaggle Input, giải nén/copy vào Working trước. Từng mẫu lỗi không làm mất mẫu thành công;
xem `status.json`, `errors.jsonl`, `summary.json`. Không có GPT evaluation.

Tham khảo: [FOA source](https://github.com/jiaxiaojunQAQ/FOA-Attack),
[SAGA](https://arxiv.org/html/2602.04356v2),
[Qwen3-VL implementation, Transformers 4.57.1](https://github.com/huggingface/transformers/blob/v4.57.1/src/transformers/models/qwen3_vl/modeling_qwen3_vl.py).
Đây là FOA + dynamic concept-conditioned region sequencing, không phải protocol SAGA nguyên bản.
Kiểm thử T4 ×2 với pretrained weights vẫn cần thực hiện trên Kaggle; notebook bàn giao không có execution outputs.

**Logic audit:** concept text được khóa trong từng mẫu; mọi checkpoint kiểm tra text SHA256,
token IDs/selected offsets, query rows và grid không thay đổi. JSON map và schedule ghi fingerprint
để đối chiếu. Dynamic chỉ đổi map theo PNG hiện tại, không sinh lại caption hoặc tự đổi concept.
Nếu muốn dùng một cụm từ chung cho cả tập, đặt `GUIDE_TEXT_OVERRIDE`; mặc định caption source
khác nhau giữa các mẫu. Schedule SAGA vẫn tăng diện tích tới toàn ảnh ở phase cuối, nên không phải
mask phân đoạn cố định quanh một object suốt 300 bước.

Progress hiện `sample x/N`, `id`, `step y/STEPS`, số mẫu thành công/lỗi và phase đang chạy
(attack/attention/saving). `status.json` cập nhật cùng progress/checkpoint. Thanh tổng đếm công việc
đã xử lý, kể cả mẫu lỗi; xem `samples`, `errors_this_session` và trạng thái cuối để xác nhận thành công.
Resume phục hồi manifest từ commit từng mẫu, kiểm tra checksum PNG/map/concept và tải đúng Qwen commit
đã lưu. Thay đổi code audit tạo fingerprint mới; run từ bản code cũ cần giữ notebook cũ để resume.


In [ ]:
from pathlib import Path

DATASET_ROOT = Path('/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset')
WORK_ROOT = Path('/kaggle/working')
OUTPUT_ROOT = WORK_ROOT / 'kltn_attack_v3_2_outputs'
REPO_DIR = WORK_ROOT / '_foa_attention_reference'
HF_HOME = Path('/kaggle/temp/foa_attention_hf_cache')
FOA_REPO_URL = 'https://github.com/jiaxiaojunQAQ/FOA-Attack.git'
FOA_REVISION = '6aa9ad463a1e812b7f77ff2e5c4b1712e7c78c4b'
KMEANS_REVISION = 'f7f36bd1cb4e3a761d73d584866d0a9c6b4d2805'

NUM_SAMPLES = 100             # None: mọi cặp có caption hợp lệ trong bảng input.
STEPS = 300
PAPER_STEPS = 300
PAPER_EPSILON = 16.0        # attack tensor trong [0,255]
PAPER_ALPHA = 1.0
PAPER_INPUT_RES = 224
PAPER_CROP_SCALE = (0.5, 0.9)
PAPER_BACKBONES = ('B16', 'B32', 'Laion')
PAPER_CLUSTER_NUMBER = 10
PAPER_SEED = 2023
ATTACK_DEVICE = 'cuda:0'
ATTENTION_DEVICE = 'cuda:1'
ATTENTION_MODE = 'dynamic'    # fixed | dynamic | baseline
NUM_PHASES = 10
HOTSPOT_TOP_K = 3           # khác với khoảng checkpoint k
HOTSPOT_IOU = 0.3
CHECKPOINT_INTERVAL = max(1, STEPS // 10)  # derived, không chỉnh riêng

CAPTION_INPUT_PATH = None   # None dùng bảng nhúng; hoặc Path tới captions.jsonl/run directory.
GUIDE_CAPTION_ROLE = 'source'
GUIDE_TEXT_OVERRIDE = None  # vd 'a panda bear'; áp dụng chung, hoặc dùng dict theo sample_id.
GUIDE_OVERRIDES = {}        # vd {'00000': 'a panda bear'}; ưu tiên hơn override chung.
EXTRA_CONCEPTS = []         # vd ['a dog', 'a toothbrush']; thêm vào source/target/guide.
EXTRA_CONCEPTS_BY_SAMPLE = {}

ATTENTION_MODEL_ID = 'Qwen/Qwen3-VL-8B-Instruct'
ATTENTION_REVISION = 'main' # commit đã resolve được ghi vào model_metadata.json
ATTENTION_LAYER = 29        # zero-based, theo config SAGA
ATTENTION_INPUT_RES = 448
MAX_CONCEPT_TOKENS = 128
CONTENT_WORDS_ONLY = True
ATTENTION_PROMPT = 'Describe this image, no longer than 25 words.'
PRUNE_TEXT_BRANCH = True    # loss FOA không dùng text branch CLIP
RUN_ATTACK = True
INSTALL_DEPENDENCIES = True
RUN_PARITY_CHECK = True
LIVE_PREVIEW = False

if ATTENTION_MODE not in {'fixed', 'dynamic', 'baseline'}:
    raise ValueError('ATTENTION_MODE must be fixed, dynamic, or baseline.')
if type(STEPS) is not int or STEPS < 1:
    raise ValueError('STEPS must be a positive integer.')
if NUM_SAMPLES is not None and (type(NUM_SAMPLES) is not int or NUM_SAMPLES < 1):
    raise ValueError('NUM_SAMPLES must be positive or None.')
if GUIDE_CAPTION_ROLE not in {'source', 'target', 'adversarial'}:
    raise ValueError('Invalid GUIDE_CAPTION_ROLE.')

# Throughput knobs. Fixed for a run; saved with the fingerprints.
FOA_AUTOCAST = True              # False: FP32 CLIP ablation; OT/K-means always FP32
GRADIENT_SCALE = 1024.0 if FOA_AUTOCAST else 1.0
OBSERVE_ALL_CONCEPTS_EVERY = 5    # checkpoint ordinals; 0=endpoints only; 1=every checkpoint
ASYNC_OBSERVATION = True         # guide remains synchronous/fresh; only auxiliary maps overlap
PLOT_FIRST_N_SAMPLES = 2         # None: every sample; 0: maps only
SHOW_FINAL_PLOTS = False
PROGRESS_INTERVAL = 30
RESUME_RUN_DIR = None            # writable existing run directory, same configuration
SAMPLE_START = 0                 # offset in caption matches; for separate dataset shards
CONTINUE_ON_ERROR = True
ZIP_COMPRESSION_LEVEL = 1

if type(OBSERVE_ALL_CONCEPTS_EVERY) is not int or OBSERVE_ALL_CONCEPTS_EVERY < 0:
    raise ValueError('OBSERVE_ALL_CONCEPTS_EVERY must be a nonnegative integer.')
if not 0 < GRADIENT_SCALE < float('inf') or PROGRESS_INTERVAL < 1:
    raise ValueError('Invalid gradient scale/progress interval.')
if PLOT_FIRST_N_SAMPLES is not None and (type(PLOT_FIRST_N_SAMPLES) is not int or PLOT_FIRST_N_SAMPLES < 0):
    raise ValueError('PLOT_FIRST_N_SAMPLES must be nonnegative or None.')
if type(SAMPLE_START) is not int or SAMPLE_START < 0:
    raise ValueError('SAMPLE_START must be a nonnegative integer.')

if not (0 <= PAPER_EPSILON <= 255) or not (0 < PAPER_ALPHA < float('inf')):
    raise ValueError('epsilon must be in [0,255] and alpha must be finite and positive.')
if type(PROGRESS_INTERVAL) is not int or not 0 <= ZIP_COMPRESSION_LEVEL <= 9:
    raise ValueError('Invalid progress interval or ZIP compression level.')
if not PAPER_BACKBONES or any(name not in {'B16', 'B32', 'Laion'} for name in PAPER_BACKBONES):
    raise ValueError('Select valid FOA backbones.')
if type(PAPER_CLUSTER_NUMBER) is not int or not 1 < PAPER_CLUSTER_NUMBER <= 49:
    raise ValueError('K-means needs 2..49 clusters for the B32 patch grid.')


In [ ]:
# Một môi trường package cho cả CLIP/FOA và Qwen. Không cài lại Torch/Torchvision.
import os, sys, subprocess, json, hashlib, time, random, shutil, ast, traceback, inspect
from datetime import datetime, timezone

PACKAGES = [
    'transformers==4.57.1', 'tokenizers==0.22.1', 'huggingface-hub==0.34.4',
    'accelerate==1.11.0', 'bitsandbytes==0.48.1', 'ftfy==6.3.1',
    'tqdm==4.67.1', 'loguru==0.7.3',
]
SETUP_DIR = WORK_ROOT / '_attention_v3_setup'

def quiet_command(command, name):
    SETUP_DIR.mkdir(parents=True, exist_ok=True)
    log_path = SETUP_DIR / name
    with log_path.open('w', encoding='utf-8') as log:
        result = subprocess.run(command, stdout=log, stderr=subprocess.STDOUT)
    if result.returncode:
        raise RuntimeError(f'Setup failed: {log_path}')

if INSTALL_DEPENDENCIES:
    if any(name in sys.modules for name in ('transformers', 'tokenizers', 'kmeans_pytorch', 'surrogates')):
        raise RuntimeError('Restart session before installing pinned packages, or set INSTALL_DEPENDENCIES=False.')
    quiet_command([sys.executable, '-m', 'pip', 'install', '-q', *PACKAGES], 'pip.log')
    quiet_command([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps',
                   f'kmeans-pytorch @ git+https://github.com/subhadarship/kmeans_pytorch.git@{KMEANS_REVISION}'], 'kmeans.log')
os.environ.update({'HF_HOME': str(HF_HOME), 'TOKENIZERS_PARALLELISM': 'false',
                   'HF_HUB_DISABLE_PROGRESS_BARS': '1', 'WANDB_MODE': 'disabled'})
HF_HOME.mkdir(parents=True, exist_ok=True)
# Dedicated checkout: không xóa/ghi đè checkout v1.2 hoặc v2 của bạn.
if not REPO_DIR.exists():
    quiet_command(['git', 'clone', FOA_REPO_URL, str(REPO_DIR)], 'git_clone.log')
if not (REPO_DIR / '.git').is_dir():
    raise RuntimeError(f'Expected a dedicated Git checkout: {REPO_DIR}')
origin = subprocess.check_output(['git', '-C', str(REPO_DIR), 'remote', 'get-url', 'origin'], text=True).strip()
if origin.rstrip('/') != FOA_REPO_URL.rstrip('/'):
    raise RuntimeError('FOA checkout origin differs from configured source.')
dirty = subprocess.check_output(['git', '-C', str(REPO_DIR), 'status', '--porcelain'], text=True)
if dirty.strip():
    raise RuntimeError('Dedicated FOA checkout has local changes; choose a new REPO_DIR.')
quiet_command(['git', '-C', str(REPO_DIR), 'checkout', '--detach', FOA_REVISION], 'git_checkout.log')
FOA_SOURCE_REVISION = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
if FOA_SOURCE_REVISION != FOA_REVISION:
    raise RuntimeError('FOA revision mismatch.')
sys.path.insert(0, str(REPO_DIR))
print('Setup ready. FOA:', FOA_SOURCE_REVISION)


In [ ]:
import contextlib, io, csv, gc, math, string, textwrap, threading, zipfile
from concurrent.futures import ThreadPoolExecutor, Future
from types import MappingProxyType
from tqdm.auto import tqdm
from typing import Dict, List, Tuple, Any, Optional, Sequence
from importlib import metadata
import numpy as np
from PIL import Image
import torch
from torch import nn, Tensor
import torch.nn.functional as F
from torchvision import datasets, transforms
from torchvision.transforms import functional as TF, InterpolationMode
from torchvision.utils import save_image
from kmeans_pytorch import kmeans
from surrogates import ClipB16FeatureExtractor, ClipB32FeatureExtractor, ClipLaionFeatureExtractor, EnsembleFeatureExtractor_ot
from transformers import AutoConfig, AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from IPython import get_ipython
from IPython.display import display, FileLink
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from loguru import logger
logger.remove()
logger.add(sys.stderr, level='WARNING')
if metadata.version('transformers') != '4.57.1':
    raise RuntimeError('This Qwen Q/K capture targets transformers 4.57.1; install pins and restart session.')

CODE_SNAPSHOTS = {}
def remember_code(name):
    shell = get_ipython()
    if shell is not None:
        CODE_SNAPSHOTS[name] = shell.history_manager.input_hist_raw[-1]

def utc_stamp():
    return datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')

def write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(payload, indent=2, ensure_ascii=False, allow_nan=False) + '\n', encoding='utf-8')
    temporary.replace(path)

def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

@contextlib.contextmanager
def preserve_rng():
    python_state, numpy_state, cpu_state = random.getstate(), np.random.get_state(), torch.get_rng_state()
    cuda_states = torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None
    try:
        yield
    finally:
        random.setstate(python_state)
        np.random.set_state(numpy_state)
        torch.set_rng_state(cpu_state)
        if cuda_states is not None:
            torch.cuda.set_rng_state_all(cuda_states)

def set_foa_seed(seed=PAPER_SEED):
    random.seed(seed); np.random.seed(seed); os.environ['PYTHONHASHSEED'] = str(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def validate_devices():
    if not torch.cuda.is_available() or torch.cuda.device_count() < 2:
        raise RuntimeError('Attack requires two GPUs: enable Kaggle T4 x2; model loading has not started.')
    devices = [torch.device(ATTACK_DEVICE), torch.device(ATTENTION_DEVICE)]
    if any(d.type != 'cuda' or d.index is None or d.index >= torch.cuda.device_count() for d in devices):
        raise ValueError('Configure two distinct available explicit CUDA devices.')
    if devices[0] == devices[1]:
        raise ValueError('FOA and Qwen must use different GPUs.')
    for d in devices:
        print(d, torch.cuda.get_device_name(d))

if 'iter_limit' not in inspect.signature(kmeans).parameters:
    raise RuntimeError('K-means lacks iter_limit; restart after the pinned install.')
direct_url = json.loads(metadata.distribution('kmeans-pytorch').read_text('direct_url.json') or '{}')
if direct_url.get('vcs_info', {}).get('commit_id') != KMEANS_REVISION:
    raise RuntimeError('K-means is not the required pinned Git revision.')
RUN_DIR = None


def concept_sha256(text):
    if not isinstance(text, str) or not text.strip():
        raise ValueError('Concept must be nonempty text.')
    return hashlib.sha256(text.encode('utf-8')).hexdigest()

def write_jsonl_atomic(path, rows):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    with temporary.open('w', encoding='utf-8') as stream:
        for row in rows:
            stream.write(json.dumps(row, ensure_ascii=False, allow_nan=False) + '\n')
        stream.flush()
        os.fsync(stream.fileno())
    temporary.replace(path)

def recover_completed_records(run_dir, selected_ids):
    """Recover complete sample commits; tolerate only a torn final manifest line."""
    run_dir = Path(run_dir)
    manifest_path = run_dir / 'manifest.jsonl'
    manifest_records = {}
    if manifest_path.is_file():
        raw = manifest_path.read_bytes()
        lines = raw.splitlines(keepends=True)
        for index, line in enumerate(lines):
            if not line.strip():
                continue
            try:
                record = json.loads(line)
            except (ValueError, UnicodeDecodeError):
                if index != len(lines) - 1 or line.endswith(b'\n'):
                    raise ValueError('Manifest has a malformed committed/middle line; refusing to discard it.')
                backup = run_dir / f'manifest_torn_{utc_stamp()}.jsonl'
                backup.write_bytes(raw)
                break
            sample_id = record['sample_id']
            if sample_id in manifest_records:
                raise ValueError(f'Duplicate manifest sample: {sample_id}')
            manifest_records[sample_id] = record
    records = {}
    samples_dir = run_dir / 'samples'
    for sample_dir in sorted(samples_dir.iterdir()) if samples_dir.is_dir() else []:
        status_path = sample_dir / 'status.json'
        if not sample_dir.is_dir() or not status_path.is_file():
            continue
        sample_status = json.loads(status_path.read_text(encoding='utf-8'))
        if sample_status.get('status') != 'complete':
            continue
        if sample_dir.name not in selected_ids:
            raise ValueError(f'Completed sample is outside selected data: {sample_dir.name}')
        record_path = sample_dir / 'manifest_record.json'
        record = (json.loads(record_path.read_text(encoding='utf-8')) if record_path.is_file()
                  else manifest_records.get(sample_dir.name))
        if record is None or record.get('sample_id') != sample_dir.name:
            raise ValueError(f'Completed sample has no valid manifest record: {sample_dir.name}')
        records[sample_dir.name] = record
    return records

def attention_token_fingerprint(metadata):
    keys = ('token_ids', 'selected_offsets', 'concept_query_rows', 'vision_positions', 'grid_shape', 'method', 'layer')
    if any(key not in metadata for key in keys):
        raise ValueError('Attention metadata lacks token-alignment proof.')
    return hashlib.sha256(json.dumps({key: metadata[key] for key in keys}, sort_keys=True).encode()).hexdigest()


def validate_completed_artifacts(run_dir, record, steps):
    run_dir = Path(run_dir)
    sample_id = record['sample_id']
    sample_dir = run_dir / 'samples' / sample_id
    guide_sha = concept_sha256(record['guide_concept'])
    token_signature = None
    for role in ('source', 'target', 'adversarial'):
        image_path = (run_dir / record[role]).resolve()
        if not image_path.is_relative_to(run_dir.resolve()):
            raise ValueError('Manifest image escapes the run directory.')
        if sha256_file(image_path) != record['image_sha256'][role]:
            raise ValueError(f'Resume image checksum changed: {sample_id}/{role}')
    for step in checkpoint_steps(steps):
        image_path = sample_dir / 'checkpoints' / f'step_{step:05d}.png'
        map_dir = sample_dir / 'attention' / f'step_{step:05d}'
        metadata = json.loads((map_dir / 'guide.json').read_text(encoding='utf-8'))
        if metadata.get('checkpoint_sha256') != sha256_file(image_path):
            raise ValueError(f'Resume checkpoint checksum changed: {sample_id}/{step}')
        if metadata.get('concept_sha256') != guide_sha or metadata.get('concept') != record['guide_concept']:
            raise ValueError(f'Resume guide concept changed: {sample_id}/{step}')
        signature = metadata.get('token_signature_sha256')
        if (not isinstance(signature, str) or signature != attention_token_fingerprint(metadata)
                or (token_signature is not None and signature != token_signature)):
            raise ValueError(f'Resume guide tokens changed: {sample_id}/{step}')
        token_signature = signature
        if metadata.get('map_sha256') != sha256_file(map_dir / 'guide.npz'):
            raise ValueError(f'Resume attention map checksum changed: {sample_id}/{step}')
        with np.load(map_dir / 'guide.npz', allow_pickle=False) as saved:
            probability, minmax = saved['probability'], saved['minmax']
            if (probability.ndim != 2 or minmax.shape != probability.shape or not np.isfinite(probability).all()
                    or probability.min() < 0 or not np.isclose(probability.sum(), 1, atol=1e-5)
                    or not np.isfinite(minmax).all() or minmax.min() < 0 or minmax.max() > 1 + 1e-5):
                raise ValueError(f'Resume attention map is invalid: {sample_id}/{step}')
    if sha256_file(sample_dir / 'checkpoints' / f'step_{steps:05d}.png') != record['image_sha256']['adversarial']:
        raise ValueError(f'Final image differs from its final checkpoint: {sample_id}')

remember_code('imports')


In [ ]:
# Frozen text table only; no image pixels or weights embedded.
REFERENCE_PROVENANCE = json.loads('{"caption_reference": "outputs/foa_attack/text_generation_v1_1_20261007T032741067291Z", "caption_input_sha256": "3be84ce065cc4486baca8f2908da5808851337499c8eb62e180528481b1678fc", "caption_records": 100, "reference_files_sha256": {"notebooks/attacks/kltn-attack-v1.2.ipynb": "47d2ed1cabf6448bb89c3363c7d3d623f1546bd4a8ffc761113f918cbdea8e22", "notebooks/attacks/kltn-attack-v2.ipynb": "e61f5918f3318951afe90baac721ca42618e5582db91010fc2b6bee6500efe52", "src/SAGA/saga/hotspots.py": "874336c3ed5903a6e691549475ef699ac75e10243c6e125c6bbd479745fc9ed2", "src/SAGA/saga/crops.py": "ae6d28de0ce1aa9fb916bde56eaee6241c2dd7ec6a2c22641e70435156d4b110", "src/SAGA/saga/text.py": "74df1f1b81ca9aa6ef0b9d7ecab6341c9ec19f8a54d92609fdde3297edd2936a"}}')
EMBEDDED_CAPTIONS = json.loads('[{"sample_id": "00000", "index": 0, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/0.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/0.jpg", "captions": {"source": "A black and white panda bear with a white nose.", "target": "A dog with a toothbrush in its mouth.", "adversarial": "A dog with a white face and black body is laying on the ground."}, "image_sha256": {"source": "3a928c42c01c6fafe680cf64746cc222dcf350a646e632940361ad9b1bc65419", "target": "9100a630bee5e0bf63125231d1dd80eb6f18677d994205240bb55e341c210c57", "adversarial": "305395e4a8dfdb3f76509dd78987ac2d391688bc4ca199346721af15d7e2348d"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00001", "index": 1, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/1.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/1.jpg", "captions": {"source": "A narrow canal with a boat on it.", "target": "A group of skiers standing on a snowy slope.", "adversarial": "A group of skiers are standing on a snowy sidewalk."}, "image_sha256": {"source": "c5b21530082c4385d0ba4be3640acc1870dbb290bbf5833e7548b5089d2dbe50", "target": "7ac8093460fcfb8f43b0414ad8488fc49be4c8e9889bd592dea8ba1b4d24c56b", "adversarial": "f4c6a026449dfd40b90cedc6430e0b26f493685497496cb81ac076f63c1e5b7c"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00002", "index": 2, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/10.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/10.jpg", "captions": {"source": "A man and a girl are in a boat, the man is wearing a red jacket and black pants.", "target": "A blue boat is docked next to a green boat.", "adversarial": "A boat with a red canopy is docked at a marina."}, "image_sha256": {"source": "f152f989c0aa71fe051bc4edc44a13ecf4dece4c22b6d9249b0c99476bdf518e", "target": "55143a19017b1a585a54a428d0c11c81061dc072f9842ac41d2717a45228742e", "adversarial": "60441c669c043cfaceee2a11ca2e8e30d373d565ccb16961851e0c9d7d4d5c8d"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00003", "index": 3, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/100.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/100.jpg", "captions": {"source": "A man is lifting weights in a gym.", "target": "A baby boy is brushing his teeth.", "adversarial": "A baby is holding a toothbrush."}, "image_sha256": {"source": "f47d75fae02ac0da2395a6d42dd600f67268de60fd3b38c0979da453f3993c34", "target": "9a98ee2b8e03a3b6b459fd2452ad9c5f5d8de8b2dd75ed86fb322c6dfffdf8d1", "adversarial": "4f54381922c25eb1df9cc529bb239cebeaee3358f1909fa1cc6d61e1e84424bc"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00004", "index": 4, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/101.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/101.jpg", "captions": {"source": "Two toasters with toast on top.", "target": "A bathroom with two sinks and a mirror.", "adversarial": "A bathroom with two sinks and a shower."}, "image_sha256": {"source": "0c6a7db229fc7668838a552e7bac304cc39cdaa7b9541074e8988cdc79916522", "target": "d1ba58a57c3146e1ca3371fd2117577e2583641abc9772abd2815bf6ab167ec3", "adversarial": "2244370e6d0abb9c5533788111bc1f6a1a797cc01312b0940ed0ed207ed82194"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00005", "index": 5, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/102.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/102.jpg", "captions": {"source": "A metal cover on a sidewalk.", "target": "A group of people riding horses and playing polo.", "adversarial": "A group of people riding horses in a circle."}, "image_sha256": {"source": "6488656de0192239cd2b7fb12c879beadbe30b5cf3aad2dba1497961a066d829", "target": "6edd6de7dbc3231a1dbd1376870f04e019376ad0fb64c65883136a44988793e8", "adversarial": "f2db170e8bfe0eeeca15d05667506c941fe6b93b492be46c4895971961b03cb8"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00006", "index": 6, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/103.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/103.jpg", "captions": {"source": "Two children pose in front of a large rock.", "target": "A train is parked at a station.", "adversarial": "A train is parked at a station with people waiting to board."}, "image_sha256": {"source": "a7b1c9c296e7269c52b4abab89345548c787f8d62fa1ec74fc4a98085af6a57b", "target": "d156cac4c3f13ba9337eded5903e7556e29138d0c206bd2b844b6399ec31e9aa", "adversarial": "4c610ffb59c29b9f94346b3ab126988e621691c68790623e8536f5326c3eff3e"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00007", "index": 7, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/104.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/104.jpg", "captions": {"source": "A green and red snake is walking on a rocky ground.", "target": "A train is parked at a station with people getting on and off.", "adversarial": "A train is on the tracks and a person is standing on the platform."}, "image_sha256": {"source": "d97c611033e335d1086b4ddb9baeb366901a6471c72ef85b8a549156d0478f8b", "target": "6cc2097f7719174db9f5a36299544aa374744cd321ec904b1985ff6217c7f597", "adversarial": "b56448eee2c324dd00d60e31d5fb75fd101c8b1a573a4a61c4ad39114f8bdf42"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00008", "index": 8, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/105.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/105.jpg", "captions": {"source": "A person wearing a watch with a black face and a white band.", "target": "A black car is parked in front of a street sign.", "adversarial": "A clock is on the side of a building."}, "image_sha256": {"source": "0f56593996b589b0004f304e75564668c74fed1d7d0a405b765ea7160019985d", "target": "f98726476274ab4f99c9f435443deaea24688fce5662920287a9f85f87db376e", "adversarial": "29ffdd0d42a96ef17463557155d876c89518d69fd79088b34eb47ee556d6ccf1"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00009", "index": 9, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/106.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/106.jpg", "captions": {"source": "A woman and a boy are in a boat, the woman is wearing a life vest.", "target": "A baseball glove with a blue cap on top.", "adversarial": "A baseball glove with a hat on top."}, "image_sha256": {"source": "9dfcd6d68ee43d1ab2610e63f049e027f60add6ab97663d6f899e21e11b566ff", "target": "70112eea0d63ebe2b29c59a126c24d7b13a8bc04f45d5ac62ede74bce84b5774", "adversarial": "2e86319e66d43f7e8b45b9d055245decadb3869f195012878937dbf62bedf1a6"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00010", "index": 10, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/107.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/107.jpg", "captions": {"source": "A girl in a red jersey is trying to block a shot by another girl.", "target": "A variety of flowers are displayed in vases.", "adversarial": "A bunch of flowers are in a vase."}, "image_sha256": {"source": "05aef1678e738f4c7f205e4c180a9b1cd65c7008263789aac8d2b1baa86fdcfe", "target": "5d03ce9bcd5be7aacb87901888486241b870ba428a789e559c60c5d507bf07a3", "adversarial": "c77c26382f3c803368e1c8553862017057d5c536354922ec21b09a4a9597d7d4"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00011", "index": 11, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/108.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/108.jpg", "captions": {"source": "A black metal railing on a sidewalk.", "target": "A bus with the number 1021 on the front is driving down a city street.", "adversarial": "A bus is parked on a street at night."}, "image_sha256": {"source": "6aa7968052c94a0bed683d8deb1fa6b143ad617fb13cc088ace6a036b2109e2a", "target": "caf45f75ba8483ac1f16e95ddea0b751e9347533dbad330573534e5cfb814498", "adversarial": "4b675c1211a280e9d94c2243ced4b1dfe788e34782c585d2bc75e70d59351672"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00012", "index": 12, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/109.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/109.jpg", "captions": {"source": "A bird is walking on the beach.", "target": "A woman is sitting on a bench with her arms raised.", "adversarial": "A person is hanging from a bench."}, "image_sha256": {"source": "bb9047d2bfa6e6a6070363cf63de5b291850191e9672f7e65c3cdf96b6a26096", "target": "f01555f19d86dc4df66c8552e3d4c5baec91f1b905b09f4bd1db9255683a3346", "adversarial": "47426e4a17e855070c198bdf8507801242d9d5738238bfc0df273c0adf2a9cbe"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00013", "index": 13, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/11.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/11.jpg", "captions": {"source": "A butterfly is perched on a pink flower.", "target": "A train is traveling through a mountainous area.", "adversarial": "A train is passing by a flower."}, "image_sha256": {"source": "f48e334da663455cb50da38548d7ad591f673342cbbd0cb416d415a714af90aa", "target": "0f74410d5f5622d74d856ca5ef622afc9da04606fbead44b2f93ef7880440fbd", "adversarial": "a6300bbbc45fce6e50193187f80d0837f18d2ad6c5757e65d14b5fe873923495"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00014", "index": 14, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/110.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/110.jpg", "captions": {"source": "A large group of people are sitting in a theater.", "target": "A man is surfing on a wave in the ocean.", "adversarial": "A man is sitting on a surfboard in the ocean."}, "image_sha256": {"source": "22435bc228365adf81c308d70808518ec33933f52116a8bfa1d2a0f3a88add91", "target": "443fa8021af93c2e1901f52fafa0f59a7eab660c542d3d1087e137f0e73eb218", "adversarial": "19cc3edb9f66afc62b52b04fe81c6dce2dea8b2df1e582e894a1557e20ff2b4a"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00015", "index": 15, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/111.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/111.jpg", "captions": {"source": "A train car with Pennsylvania on the side.", "target": "A cow is walking down a street in a foreign country.", "adversarial": "A cow is walking on a street in front of a building."}, "image_sha256": {"source": "f62f7eaa7fca5b6d1a1c1a58d7faa9ec3af5cfb0c337955173c75cd0a195ae08", "target": "92ca7072f76196c19713ab2823d5175a5cc6e85f903682156502834ef66b2625", "adversarial": "e478326044cedd5d8c432072ccb2d81642c0bdd8a109863a263cdd82ab704e02"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00016", "index": 16, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/112.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/112.jpg", "captions": {"source": "A blue and white airplane with a man standing in the doorway.", "target": "A white toilet is next to a stone counter.", "adversarial": "A white toilet is next to a white sink."}, "image_sha256": {"source": "d445f4a85d21cfa08cf11bb22a00b3d47100fb7801d8694264ca4d9f03157d3f", "target": "384fd89c73eb0ae5c11379747df53e17ef146d48f22bc1171c7e38d53f3aae07", "adversarial": "6b26352c345cd1b15d2bbbdb70016b51cfdcb0b0ac3b0d4462ad6c381f0597cb"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00017", "index": 17, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/113.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/113.jpg", "captions": {"source": "A blue car with a chrome grill and a clock on the front.", "target": "A large building with a clock tower and a steeple.", "adversarial": "A large building with a clock on the side."}, "image_sha256": {"source": "1aacc5b9ecada18262d0c90543058481dafcaa1c2cd1116ee9cc92295de9c616", "target": "d01f7c594f3f1b6827d5a377fd968cfeb9b1c62a90721d013b00a4788c681a5b", "adversarial": "3cb1703f5e212f3f638ac9800dff5fafc893835da82e030c8fb470595e613dc1"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00018", "index": 18, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/114.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/114.jpg", "captions": {"source": "A large assortment of donuts are displayed in a bakery.", "target": "A white toilet is in a bathroom with a white tile wall.", "adversarial": "A bathroom with a wooden door and a shower."}, "image_sha256": {"source": "33d0ef0f005093ab94230fb96a3dc848228be41e62e9746aa2e1a478b6c1f371", "target": "2984abb2d846c6fc9437a234597f05946fdfd364eaf2a8414f89312e6ca56a18", "adversarial": "cd5a797462a0a1c9b3b393abb0cd02b747959f4be9b3f28e93eb22c9cfa27985"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00019", "index": 19, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/115.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/115.jpg", "captions": {"source": "A large building with a dome and a gold statue.", "target": "A herd of zebras drinking water.", "adversarial": "A group of zebras are standing in a building."}, "image_sha256": {"source": "5d8c4a4ffa3ad71076cafd152fda56364aef51c56454542cd213725db2d80f6a", "target": "48693a4c6eb39fc7165b34696ae99a87a3c823987b2f497124c14d17b25e07c9", "adversarial": "beb9e4416f096ebf4d6d90caf975229d88111715919cb2dc2921ac892d662b7a"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00020", "index": 20, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/116.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/116.jpg", "captions": {"source": "A group of young men playing rugby on a field.", "target": "A stop sign is next to a shopping cart.", "adversarial": "A group of people are walking with shopping carts."}, "image_sha256": {"source": "89c40a1d4d77d2de6a48fa04311d26aadd2bba26a3823d795e42c789afc8acf2", "target": "0f348be4e48cc2e13a8a25f2cc38488e45d343defc490acbc703e0fea2681626", "adversarial": "64b156bc0d13dd058e8999447726133e219bd354cb33c0964aeeb44585569e39"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00021", "index": 21, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/117.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/117.jpg", "captions": {"source": "A man with a beard and a guitar.", "target": "A giraffe is walking through a grassy field.", "adversarial": "A giraffe is standing in front of a tree."}, "image_sha256": {"source": "412e38ea2534f3602fa7dbd688efeac517b1b1f7d63984e4b6ca0c767a912741", "target": "c08b2e368bccee101b09c2ba32c09450fcf34f332bfbba038518cacbe97187b8", "adversarial": "4134b778383f8f8efa2b8067ce8ed55d72e45fc5c9494602191368067c2bc6c6"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00022", "index": 22, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/118.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/118.jpg", "captions": {"source": "A group of men in uniforms are standing in a line.", "target": "A colorful bus with the word \\"Jakoo\\" on it.", "adversarial": "A bus with the word \\"Jako\\" on it."}, "image_sha256": {"source": "2e95007e0e51d9bdde65e0b4c7b601d904c6e4ae3944ec13ef5cc9c834f2b4cc", "target": "17e8ae0f7177c0a1e966bd4e0e17b3102c09c760894daa10f9bab5e372c3fd6b", "adversarial": "f4500b8e1bb28892e78c7e50cdb34bf5282b530679d1fd816e5b9c5c0033a0b9"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00023", "index": 23, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/119.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/119.jpg", "captions": {"source": "A bear is laying in the grass next to a rock.", "target": "A yellow CSX train is traveling down the tracks.", "adversarial": "A train is on a track in a park."}, "image_sha256": {"source": "b0eff4f19e3b21a97fa6267f82e6ae69ada3f4b08129d934e18dd2dbf9fce93b", "target": "c08a6f0d7c7c8a1577717e6c7b0ec0bf3c6b83a3de0522ffb69e068274a90154", "adversarial": "aac8fe6fb624e81f646e42416024cfd4161662a24e2d8a66a374dad7b973ff01"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00024", "index": 24, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/12.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/12.jpg", "captions": {"source": "A piano is in a room with a statue and a painting.", "target": "Two white horses with red feathers on their heads are harnessed to a carriage.", "adversarial": "A horse with a white mane and black tail is standing next to another horse."}, "image_sha256": {"source": "c34ef3b9363ad4ba1025f553c13ecd75b669191556ed2ead69ca8cb5d8d023f1", "target": "5bd68479992fa2ad6957be291c5b7d46d6219ba277b6291813b7d6a5cbcedca4", "adversarial": "0d49a7fa98f1e09883565e92521807296e00dccf1dad6d5565aa2d9de3aacaa1"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00025", "index": 25, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/120.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/120.jpg", "captions": {"source": "A crocodile is laying in the water.", "target": "A white bird is perched on a wooden table.", "adversarial": "A bird with a black beak is standing on a carpet."}, "image_sha256": {"source": "34a22d4f7570374ada0347c3ec91de1824ce107226f1ff6fdd1cfac895d0699e", "target": "8aa6c9f5511d1c80de9c8c465c864077ece09e21f02e6f2d14692b22b575d675", "adversarial": "d8ed9fe4508e9f77923f78807891cb731ebcb2d3addcf9e14ac6725387094e62"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00026", "index": 26, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/121.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/121.jpg", "captions": {"source": "A black and white insect is perched on a green stem.", "target": "A street sign is on a pole with a tree in the background.", "adversarial": "A street sign is hanging from a pole."}, "image_sha256": {"source": "8c15fbab68865026b025f4a4e77c885894663b8223882c042626984777747474", "target": "98ebcf18c45f87443c2a09a7b37f766d79bdfaf72ed6899e8f402762273092c3", "adversarial": "9a93b1e67dfb1bd6337f223e3254cb4d52da2c69e8a8bd61781c100caa26d1c8"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00027", "index": 27, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/122.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/122.jpg", "captions": {"source": "A turtle is laying on a rock in the ocean.", "target": "A plane flies over a beach with people swimming and enjoying the ocean.", "adversarial": "A plane is flying over a body of water."}, "image_sha256": {"source": "d8796d58b7d07b772d4d58d337901a0caa653de8d7f3bd2e23f11ca070864a7e", "target": "8475b1b6709113ba95973fb2638677137f8736eff5bb66db529dae4c710d42be", "adversarial": "83f182b2600afcae86dd7920d97019393a842eaa5b1016faf3f561a2d4c784c8"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00028", "index": 28, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/123.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/123.jpg", "captions": {"source": "A statue of a woman holding a book.", "target": "A man is performing a trick on a skateboard in front of a statue.", "adversarial": "A statue of a man on a skateboard."}, "image_sha256": {"source": "ad66c8d5eaf1b43c0a308b11e40a4392ae7240ca0054b5f12a7a57bcd30b8d09", "target": "b2823aa9d8f7eb32a36821d5a691aaa947a5d66152b8893b966172569cd7b00b", "adversarial": "e83a95f4b0c0e9017cd497e4be196bf2ce73d5140c05c903da6e210e43d7d90e"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00029", "index": 29, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/124.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/124.jpg", "captions": {"source": "A mirror shot of a street with cars and people.", "target": "A blue sky with a white cloud and a plane flying in the sky.", "adversarial": "A reflection of a building in a mirror."}, "image_sha256": {"source": "f0e8c0e6bf9323238203612861634aec48d03d3f754fd463168836207d383912", "target": "96f5eecfc2d3a93555eb3575af848be1a8f5fcbc2e382fd7e465799338bbd6ae", "adversarial": "ae0a2e977f7a5cebc47aef09f4393519ab459b0a430bf6bc8a8e91c2369cce99"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00030", "index": 30, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/125.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/125.jpg", "captions": {"source": "A cup of tea is on a table with a spoon in it.", "target": "A woman in a wetsuit is surfing on a wave.", "adversarial": "A woman in a wetsuit is surfing on a wave."}, "image_sha256": {"source": "b0313890c6b00c12764abe10b63bbcf707bde892b687aa8214091ec301ac0a5d", "target": "26a5bc3927b2d7135e92191b88a5a3b8acca6b52ac7f5eb49c8b9ec12572d364", "adversarial": "eab89148683ce65a766ab02325b58305e07259fbf7fdbf7c0c9af4592f7ae7d3"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00031", "index": 31, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/126.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/126.jpg", "captions": {"source": "A fly is sitting on a leaf.", "target": "A bathroom with a sink, mirror, and toilet.", "adversarial": "A bathroom with a green tiled wall and a mirror."}, "image_sha256": {"source": "d28e457a6ccb3bf3fab6a07662bc0911dad96dbc739ee302755f6501bfb483ec", "target": "c04d0ca9941598531e9ab542c2c06f00e86f2ca87eb6f552a0db139dd7beda97", "adversarial": "a2e75a69d915c14373c2da2e035c1bb54af39d5f7ac1de5962e974fc7cdd5433"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00032", "index": 32, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/127.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/127.jpg", "captions": {"source": "A tree with pink flowers and green leaves.", "target": "A train is on the tracks and is about to pass under a bridge.", "adversarial": "A train is traveling down the tracks."}, "image_sha256": {"source": "42d0d2b1e4def1331681a7e3ef83a6b04cdf3c618ad2b207956e5bce7c4ed1f7", "target": "ce30ed70c2d4c77f65eea2637a1a7c646bd1332275c69a30af521c4198f527c8", "adversarial": "c2a88ff42636d88b9a9b77d2169969f2273fdcf4db728e9dde30408f943d52fb"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00033", "index": 33, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/128.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/128.jpg", "captions": {"source": "A red car is parked on a red carpet.", "target": "A small kitchen with a table and chairs.", "adversarial": "A kitchen with a car and a dishwasher."}, "image_sha256": {"source": "2cfc53570b9874f0b6248a3646712b8b41ffa6880184b17634f2131630c52e45", "target": "66e2325c4f4e5827b5e55b59ce7ed46f9b7d4b706efa78eed862840e8ba83e42", "adversarial": "f664a10a022baa5b01d2d5a7674fe43504907e4a0fd1a673977befa568aa1e11"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00034", "index": 34, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/129.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/129.jpg", "captions": {"source": "A lit match is on a black background.", "target": "A blue train with people on the sidewalk.", "adversarial": "A blue tram is on a track in front of a building."}, "image_sha256": {"source": "47178d60c302a24ee09b34ac01a3a03ceae709d5716808d5f67aefb8352bcbf5", "target": "9fe6063b1d95da850ea723c944025e7d947306a05627546422c3c285e4d5962d", "adversarial": "aaee9df3b3a87a1a4f98f0ce67b2ba67ce909046e93fb7310918d8fa02d20f22"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00035", "index": 35, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/13.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/13.jpg", "captions": {"source": "A lake with a rocky shore and a small amount of water.", "target": "A person is cutting a cake on a glass plate.", "adversarial": "A cake with a fork in it."}, "image_sha256": {"source": "d499ffad5d73739aaf6e3bf3e3d4d80b19432cb11ed54c34923a4f7524d853d8", "target": "c1b4433f02155aa908e94ee7bd59f12cc7c33e6ac49cd4a45653a54d903908c0", "adversarial": "cc1fe63555c2b9d1729d62aa786722bc1b12903c6e1cec24afb3f1e572848e72"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00036", "index": 36, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/130.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/130.jpg", "captions": {"source": "A pig with a black spot on its ear is looking at the camera.", "target": "A black kitten is laying in a bag.", "adversarial": "A black kitten is sitting in a bag."}, "image_sha256": {"source": "42b24a1f05149ed45b7b2a83ac5baeb7ccf6fb3fc0f2a287f2c83d37e2b378b0", "target": "b27b32d7738ea025b37e4ef711aa28e6a9bf8b66d9e7e4c2a4fdb19e116779f2", "adversarial": "81f5817ebae3e7cfb01aee1592e1ef6bc0e69c60254ebe70306696c65fbcf01f"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00037", "index": 37, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/131.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/131.jpg", "captions": {"source": "A large market with a variety of fruits and vegetables.", "target": "A church with a green roof and a clock tower.", "adversarial": "A view of a city with a clock tower."}, "image_sha256": {"source": "34c61a8807b514be393d774db31824433c5585256b13f62ee85355113bf50ab9", "target": "b5b618af9646d4eda7833cdbc88b7715b38f8cfedbba1102a4a417c774fb2982", "adversarial": "0eb9b8939799fc6555b8ea51392a6a29c812eb1f17d6ac9fcde3965772b36611"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00038", "index": 38, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/132.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/132.jpg", "captions": {"source": "People walking in front of a building with a large window.", "target": "A group of people are standing under a tent on a beach.", "adversarial": "A group of people are standing on a beach."}, "image_sha256": {"source": "1db12c532900e7bef7fea50a6336f32f56d1ffd8e286c017905dedc80df068b8", "target": "291bd2a5ef9f3cdeaff143e6993f3438f61abf4e12046fe8a542159d066ce28c", "adversarial": "6a05024b93e37096ac2c0ab2e771c8c9e0fc1ca9420d82b994e430481f426b9d"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00039", "index": 39, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/133.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/133.jpg", "captions": {"source": "A 30 license plate with a blue background.", "target": "A green truck with a wooden bed is driving down a street.", "adversarial": "A truck with the word BOOM on the side."}, "image_sha256": {"source": "78e5a2ee08c0552f7ca3d822cf49ecf9b5ad7255217f06dec6d123fb836a548d", "target": "651047b01dff38cc03e6b34ce1e259b443452bb1c45a6eaef5d300d055ca7e52", "adversarial": "00e8a94496a25d88848080deabaddfc8115b4e20500041a7420bd86978219959"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00040", "index": 40, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/134.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/134.jpg", "captions": {"source": "A hot spring with blue water and steam.", "target": "A glazed donut sits on a paper bag next to a cup of coffee.", "adversarial": "A donut is on a table with a cup of coffee."}, "image_sha256": {"source": "41fb7ec37e54efc362b5da804cf9d9a67b8ba6430dc1f879a0ca36ab09a4d68b", "target": "c947dc01020a16653ec3734b15255e7edab8873ca3f38635e03f0c47de47d6fd", "adversarial": "714a593f07aab0516d9ca372ca955cf2e350ea9432bdf3c6c58009035797655d"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00041", "index": 41, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/135.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/135.jpg", "captions": {"source": "A dead plant with a red bug on it.", "target": "A white dog with a black nose is looking out the window.", "adversarial": "A dog with a white coat is looking at the camera."}, "image_sha256": {"source": "0506f9080ae3622eac99549e76a7b6dadfa8cc599338b7075dd6291ec43b555f", "target": "42c4ab964f59b5f6c29db135d00a415fe6aa390fa070ee12d914fc60c3097c37", "adversarial": "12d93e592b3094aef7f16eff6ba1e4df86253c8f7adfc5caf398e3c2b9745a1e"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00042", "index": 42, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/136.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/136.jpg", "captions": {"source": "A piano keyboard with white keys.", "target": "A church with a clock tower and a yellow building.", "adversarial": "A building with a clock on the side."}, "image_sha256": {"source": "a719275e3504c4f48ce45e495db44e3c6eed18133947c047c4a950517da9562a", "target": "76a8639194c5610ddd4af3a6e080dfb96cb058e3075048b255583e6984f08eac", "adversarial": "811a7eb21f79d24c0916132368aef94ea7a131799144594b6945166c38266be2"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00043", "index": 43, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/137.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/137.jpg", "captions": {"source": "A washing machine with a black door.", "target": "Boats are docked in a harbor.", "adversarial": "A round window with a reflection of boats in the water."}, "image_sha256": {"source": "179c81fcc81310dda43c5feb1d4270a9139c4544a71bff46a0b8a40b2ced685b", "target": "c96835e6ef885bcca9294f2acbb6a2b3163ccf7dadc29a1b3d26ce4a6aa701b8", "adversarial": "ed2e80568347d39130ab6e711568af4f4e35955056b1cebd8e507ede188dd374"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00044", "index": 44, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/138.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/138.jpg", "captions": {"source": "A group of sea creatures are seen in a coral reef.", "target": "A red truck is parked in front of a house.", "adversarial": "A small plane is parked in a field."}, "image_sha256": {"source": "f59e9464bab18a9140be18f6fd522787b5df19fdfb2189feb5f2c838ebf108a5", "target": "72396662ce0f7cd15f28df36e684de914f7bc2c16272c9ce050a63dedc8ffd7b", "adversarial": "5e2bf92aedf305884b7a3ef9c69c35b92713580f48c5ab91bfdc84246a6eabc6"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00045", "index": 45, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/139.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/139.jpg", "captions": {"source": "A First Niagara ATM machine with a sign that says \\"This ATM is hasle free!\\"", "target": "A herd of elephants are walking through the woods.", "adversarial": "A herd of elephants are standing in front of a sign."}, "image_sha256": {"source": "799011942be69b0f34ce71948b48507a4d2799c810026212814ab22b6a883350", "target": "fecb7cc924de8224ff349f3172d53aded548d061ede45575b95bbde96dd0a5c2", "adversarial": "befd321416cbe9bcaefdb37ebadd8ec5a913cec322f49286bfa98e033fa11b3a"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00046", "index": 46, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/14.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/14.jpg", "captions": {"source": "A large library with many books and a chandelier.", "target": "A man riding a horse on a street.", "adversarial": "A horse is tied to a post in front of a building."}, "image_sha256": {"source": "9777b95dbb178cce9178baa03d439d6e56e0f94b5fbfa6c752515b743453dd85", "target": "1048c9195bde612b4a8d9cac496c479e3d9edbf30e0d629d04c4b27c1d5447d9", "adversarial": "6931a9660147f8b8b24c6d7318f8357869cb3bae68c4131cae1e963d09aa5cac"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00047", "index": 47, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/140.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/140.jpg", "captions": {"source": "A bee is flying near a flower.", "target": "A refrigerator with a bottle of beer on the door.", "adversarial": "A refrigerator with a lot of food in it."}, "image_sha256": {"source": "9c2c3079248d288a6be2ac9671b4aed148840be5dbb2b152e787054e81a41df6", "target": "8a36a86e2a8c8cedeffe550f84f29c707553a66552b848c3191c5902c058e58e", "adversarial": "fa4408cce96d354bf992ffac237f5a6c35946d283be54f25130e2f1793363717"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00048", "index": 48, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/141.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/141.jpg", "captions": {"source": "A chandelier with many lights on it.", "target": "A baseball player is swinging a bat at a ball.", "adversarial": "A baseball player is swinging a bat at a ball."}, "image_sha256": {"source": "f3617311477034b022d2160c3abccfff15c0d5865e47376e3a80b53fe1ad36e5", "target": "497eac3996a786f4ce16125b3e1f95764e4652b4455c9d6f80fbc4a240146b21", "adversarial": "900a6b778223e068559dd6b49424583b0e0f0d5c73f23abdacadfa5adcdfe432"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00049", "index": 49, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/142.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/142.jpg", "captions": {"source": "A red worm is laying on a rocky ground.", "target": "A breakfast plate with eggs and pancakes.", "adversarial": "A plate of food with a fork and a knife on it."}, "image_sha256": {"source": "4dd9b59b0782a25c9c34602411cbbf47ef8f4e7fb30c6426ea4b41d22d521358", "target": "b9ea6d30a38524df08d50735e3cb76a61780066574b69e3f1a42c7a398ff5b1c", "adversarial": "a313a5bece5dd18f9615369be08da6eee744fc63857d755399231c2ddedc6664"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00050", "index": 50, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/143.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/143.jpg", "captions": {"source": "A person is driving a car with a license plate on the windshield.", "target": "A red and white sign is on a pole in front of a building.", "adversarial": "A blurry photo of a street with a stop sign."}, "image_sha256": {"source": "6dae5c4a0a2de17a32bfb89250604b8f5b6805448ba019811a0147ff0eb299d4", "target": "d1973baf15e3cf34d845fd05cc577964383ba7f39ac618ed5f49c04af39f4d44", "adversarial": "9b293d63fe153f75c219729f04d83a8c79f8574b8f716cf2eafe79204a408013"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00051", "index": 51, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/144.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/144.jpg", "captions": {"source": "A bubble is blown by a tree.", "target": "A kitchen with a window and a blue wall.", "adversarial": "A kitchen with a tea kettle and a wok."}, "image_sha256": {"source": "62cae0a487f7b176ae4d1846be97c50a141c9bcdb897253b79438e1bebbe7822", "target": "e06f61caf68df9c67f435d9790b87aa1a37f1332eef450a5534bf608037a6ec3", "adversarial": "d7d6911b3aa79671498e06284b581b0f7233e3ec622f505988607110e83816fa"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00052", "index": 52, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/145.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/145.jpg", "captions": {"source": "A box with a fake photo of the moon landing.", "target": "A group of giraffes are walking through a forest.", "adversarial": "A giraffe is standing on a fence."}, "image_sha256": {"source": "2f5050b3f10dc4b1b9b9665aa341a73bf54f50fc0e386f277ed6791cc804b11d", "target": "a90c8f9f6ac009efafe8788b53745fff17b5dbc6e91ba5e9dec8050b6d3e94d1", "adversarial": "29045086945f26a76a9505a2fb1149056a273c795a91508a69b19279c79ca70e"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00053", "index": 53, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/146.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/146.jpg", "captions": {"source": "Two old fashioned telephones are mounted on a wall.", "target": "Two blue and yellow trains are parked side by side.", "adversarial": "Two trains are parked side by side."}, "image_sha256": {"source": "b21f20f9d601be5d4d7fd3f7fc59b00b7ee80f9c7822627b3739d69093407615", "target": "7de805a894696b63fc1190d02c3aa6bce134fcd9364ddf9576c031917d0d93f2", "adversarial": "15ead17beb5a5aae3fdaafb9cf17b9c5fc73f083adcc3f6ac3ecf8879bf2725c"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00054", "index": 54, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/147.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/147.jpg", "captions": {"source": "A fountain in front of a building with a flag on top.", "target": "A soldier is standing on a military vehicle.", "adversarial": "A man is sitting on a tank."}, "image_sha256": {"source": "ac925758188d5ce4d2877dbab1302ffee07c2d583c159e21acd2de784b65a682", "target": "5baf542fa40f4e68c5194140167fd41ec3726e509834c263129e5c11e2f159dd", "adversarial": "6eb8eb23b1c71aa9be9f1c8ba3c5670ce7f4d3e233b4ae47e99a8ee3ae097cdb"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00055", "index": 55, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/148.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/148.jpg", "captions": {"source": "A pan of chicken with toothpicks in it.", "target": "A man in a wetsuit is surfing on a wave.", "adversarial": "A man is surfing on a wave with a surfboard."}, "image_sha256": {"source": "78ab2333bd815f8b034bef4ad3df462d79e12396e7aadadbe0ef0a0b0e3d153b", "target": "f6a5d6f35301bd54a80b4e4de91bce8d0e94dc1e869ed4038d1438e75ef8e2c9", "adversarial": "1bdfcca290e9e4efbf4634b846b12511a25d2ab052466712876d2b6175ec536f"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00056", "index": 56, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/149.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/149.jpg", "captions": {"source": "A red car is parked next to a green car.", "target": "A man and woman ride on the back of an elephant.", "adversarial": "A man and woman are riding in a car with an elephant."}, "image_sha256": {"source": "4a361b765479ea11925362ea5e0a7467f0c95937fd77de0097e64554f307498b", "target": "9d70c0833c417b11b7a38d5a286b0eebce9036e75a361ca7d65fe86c916dc82a", "adversarial": "f7787b3a6e63b64ac01f709e6df23a09a31b1cdc0dd999b8d6c5ed66c940d454"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00057", "index": 57, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/15.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/15.jpg", "captions": {"source": "A bird with a blue and black wing is perched on a rock.", "target": "A man in a costume is talking on a cell phone.", "adversarial": "A man in a medieval costume is holding a drink."}, "image_sha256": {"source": "b4ee24ed9cd5cc76245869eab1ab20ae518f8e1b34952030778e1e451a774bbe", "target": "736f895b53c7c3f3029d1ac10e17c55630e11d6b38ac47c49a2cbd572339320f", "adversarial": "5835048c21325f6d91e8484021d2dea1803805f1b70be4340817e338f6228b5e"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00058", "index": 58, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/150.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/150.jpg", "captions": {"source": "A house with a thatched roof and a doorway.", "target": "A table full of vegetables including broccoli, onions, and potatoes.", "adversarial": "A bunch of broccoli is on a table."}, "image_sha256": {"source": "722472104aa768f519d920ef908958a2d63bcec19e618b02112826e01ebf93f2", "target": "f1106b2afc7fca246622dcbe3d6ca5933061ccab2bb29ee5044bba51ee3824bc", "adversarial": "4ff21e9b795f7d633d99d109b7cade4f0b3a4c7c809eb2a2da0984f5fd3b4963"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00059", "index": 59, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/151.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/151.jpg", "captions": {"source": "A black car with a yellow rim.", "target": "A yellow and black train is parked at a station.", "adversarial": "A train is on the tracks and is reflected in the windows."}, "image_sha256": {"source": "49e398c33fcbf1eeff82f9f2efbd7f689965e282053d5009cef1df51510abe54", "target": "fb0ac6c26acfc17d0776fce0c4f76c0ce7d00638baefcab3fc2a52910e275bae", "adversarial": "db8270c3b1e58636be46c5d119791dc8c556239fa44e7445c1b25429f8b9673b"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00060", "index": 60, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/152.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/152.jpg", "captions": {"source": "A wooden fence with a post in the middle.", "target": "A giraffe is standing in front of a wooden fence.", "adversarial": "A giraffe is seen eating leaves from a tree."}, "image_sha256": {"source": "b19267de77ef7ffb7388e8a8ca5cff57f347e2735134d262b26e60eb7839cf69", "target": "d637e2723ad07c6a510c1e8a0374493006bdb42797f8f7e4d92c57b650ff1db8", "adversarial": "8888702014e15ac8d53bccc8435346edea2b079f738da0a3970a1e951a32218c"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00061", "index": 61, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/153.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/153.jpg", "captions": {"source": "A painting of many umbrellas in different colors.", "target": "A clock tower with the name \\"The Bayview Bank\\" on it.", "adversarial": "A colorful clock with a picture of a city on it."}, "image_sha256": {"source": "cba4c1caa0fce5326df3ab4d9b85a4edc751d8645b902dc16205d2c3e2c2f63f", "target": "99f661e959a15f5c2eddb86d345e1a7f9b581ec476e68ddcae4bf201c7a36e57", "adversarial": "c7eb105c3d0ca302011cfb7f9be808fd28044a8cadfe32ca2c30830b2c5845b2"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00062", "index": 62, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/154.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/154.jpg", "captions": {"source": "A brown and yellow bug is sitting on a leaf.", "target": "A woman is lighting a hot dog on a plate.", "adversarial": "A woman is holding a piece of bread in her hands."}, "image_sha256": {"source": "9751982db9fa72e7496d9141cef965432188a6684a4583b3c0c07fafda5e26ea", "target": "f65d5eeab209da1b49555049443270a8a5bfd3f092104d735408f202a72151e7", "adversarial": "4ba8993af7b87e26df104bd7f3eafa537654105ce8f4577c91fd47a8f8ad717c"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00063", "index": 63, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/155.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/155.jpg", "captions": {"source": "A lit candle in a white candle holder.", "target": "A motorcycle rider is on a city street at night.", "adversarial": "A motorcycle is parked on the side of the road."}, "image_sha256": {"source": "e88d3c19a0fdbe12a59c9b19cd2c7b31b19e9ed5e03a1851ce46f27b42547b66", "target": "b5c715c2723cac4b7587c8fdbb315436ec20f361e9d5cfdb7e66a36f8556d69a", "adversarial": "379070cdc8acc06787ae898d4d12c25085932e2d8cdb0d1795153fdc0bb0860e"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00064", "index": 64, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/156.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/156.jpg", "captions": {"source": "A man in a tuxedo is talking to an older woman.", "target": "A man in a blue shirt and a woman in a white shirt are playing frisbee.", "adversarial": "A man in a black shirt and white pants is playing frisbee."}, "image_sha256": {"source": "5305f253c5e6f8dfeb0f21a612e0cc3a2247f14dd09e51588208dbbbdbb5a43d", "target": "e2d65e10099ed52923ab97d8d113f46f0fa8100a9c37a1f93edbe2b6f5262eb8", "adversarial": "314a0cd3342e43b9d6ccda3082ebabdb91c7c993cd430c308f6a6c789f5c67fa"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00065", "index": 65, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/157.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/157.jpg", "captions": {"source": "Two otters are sitting on a rock.", "target": "A kitchen with a stove top oven and a microwave above it.", "adversarial": "A kitchen with a stove and a refrigerator."}, "image_sha256": {"source": "df77a29be351c14143d297a7998705ef109d75f9bfae9cecdae1329e95f50999", "target": "0ec0e58e40c8810478cbff184afb119bbd2b5152d51c8216b90c78b2976afd10", "adversarial": "52afcee49a2db3f28171cc766b120fc43df47afa89d8f8b224650ef0d0ce7ed3"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00066", "index": 66, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/158.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/158.jpg", "captions": {"source": "A red and black caterpillar is on the ground.", "target": "Two children play a video game while a woman watches.", "adversarial": "A child is playing a video game with a remote controller."}, "image_sha256": {"source": "4777b61ea6c245421bfbb7ddf107d327dd74f6b034b8f1009c3c0c9b3355209f", "target": "a292b016ca8ead0a92baae9ec8ef278d3340147f01a33072aadb1878694420fa", "adversarial": "878c8263284577b2c43dc4f6c0bd3ed003bade9542dffae0691e5088010b8525"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00067", "index": 67, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/159.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/159.jpg", "captions": {"source": "A person is painting a colorful rock.", "target": "A man is performing a trick on a skateboard.", "adversarial": "A skateboarder is captured in a time-lapse photo, performing a trick on a ramp."}, "image_sha256": {"source": "f45f00d259101a53d534436fb2aacce64afc2a8a47711f9eb4bea6dc65df462a", "target": "ca222719a7228d064e5f8075dfee97dc8b36133e0e817235d488acd64869c65b", "adversarial": "d19230c95cb4d681a859799263ee20950f583d50fbeb18ea18c6646dc15236c0"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00068", "index": 68, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/16.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/16.jpg", "captions": {"source": "A large church with a clock tower and a courtyard.", "target": "A basket of oranges and lemons.", "adversarial": "A bunch of oranges are in a basket."}, "image_sha256": {"source": "f2444e13cf8e23e348001f5468c65227222ddb1e46a92181dcb8a78572572632", "target": "e7031096208bd858ed7c4a207a443c053535746cfd71646a170f663ba48185f6", "adversarial": "92f7305df531552e6bc15b892d58fa2045c6ecc94102ee5d1bfd752f4e8c6abe"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00069", "index": 69, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/160.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/160.jpg", "captions": {"source": "A tall tower with bells on the top.", "target": "A man wearing a red shirt and glasses.", "adversarial": "A man wearing glasses is standing in front of a building."}, "image_sha256": {"source": "c90bd0a77f6a6ffef2ed0c9355624b25a923aabd38c650b0cae86558237f5e99", "target": "c8ea0034640d6b4ee706675f1a8241cefc29ad9b801751bb6da611bf749b097c", "adversarial": "cad6c3ac6b8004855aff36fef40acb08fae98222df05165f95ee857d9e32e847"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00070", "index": 70, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/161.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/161.jpg", "captions": {"source": "Three blue jellyfish are in a tank.", "target": "A hot dog with ketchup and mustard on a bun.", "adversarial": "A hot dog is being held by a person."}, "image_sha256": {"source": "2d68b53b53219cd57063fabd627579ae56335781b1517350c25826c21a8bf794", "target": "f7b54b1c83a1f2dc1ed86a978fc7407c4a8ab482df1e5a7e1a4797deb16e5f44", "adversarial": "690aecd9a9fc2d009706bfb94f0a2d994f2eb96029fc3884ba7b239268532ce4"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00071", "index": 71, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/162.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/162.jpg", "captions": {"source": "A blue and green calendar with Arabic writing on it.", "target": "A bicycle is parked in front of a store.", "adversarial": "A colorful mural of bicycles is painted on the side of a building."}, "image_sha256": {"source": "facd5dde9ed2737a1d68cb1b0c354d7ae594f920506ac9fd9921fc5a46c7f11b", "target": "fed998bb8559191f29639b5f59a29627966ad12964eff9edfaa1dcd520089e33", "adversarial": "d1898bf593df8d546985e54944574d7ff319069d09d33f85749846c4a09ef727"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00072", "index": 72, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/163.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/163.jpg", "captions": {"source": "A mountain with snow on it.", "target": "A woman and two children are sitting at a table eating hot dogs.", "adversarial": "A woman and two children are sitting at a table with a hot dog in front of them."}, "image_sha256": {"source": "d9915ac371bfde718aae947d4ae6a3380ddb261f5864cac992097671a524959a", "target": "bd88ab338b4efc479b4edfd32660b0411bee816cc10ca5fa4079eb02e5828c82", "adversarial": "59fe9b2aa91a081a61af1d90c015599348e5b34cd7f0ccca02d580a37e368585"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00073", "index": 73, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/164.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/164.jpg", "captions": {"source": "A black car seat with a black seat belt.", "target": "A brown cow is standing next to a black and white cow.", "adversarial": "Two cows are standing in a field."}, "image_sha256": {"source": "4361e1eb42f43635cd679af2be0f579e24679414cd49ce3aef08a87ccf1aca29", "target": "187868c79c935254a1da144196d6209b5a40f3e87adea6814bcf9eeecdcd3300", "adversarial": "2b6d8aa7109e83e699789c9028c46e3e2907d6f6d902dd82982243d0087a206e"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00074", "index": 74, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/165.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/165.jpg", "captions": {"source": "A plate of food with a beer and a glass of beer.", "target": "A giraffe is walking in a grassy field.", "adversarial": "A giraffe is standing in front of a wooden structure."}, "image_sha256": {"source": "2aa8165d72632ea0b09cc38bef99190c43fc7983aad72110a4e1ec349d480588", "target": "45bfd30121a4eb9cc2f9e1615fe7e567d5724a29cfe26768ca207d3aa8b9581f", "adversarial": "700e235004c277408b35bd2a0bd874d2b2d0b7a58e1d74520ba846b23171e26a"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00075", "index": 75, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/166.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/166.jpg", "captions": {"source": "A purple car with a wood panel on the back.", "target": "A bathroom with a white sink and a green bowl.", "adversarial": "A bathroom with a toilet, sink, and shower."}, "image_sha256": {"source": "7bf54f63673218c90e513948cf2ed385eb2955cfc90b41a4014384f1ba7539f8", "target": "9a0d3be17031188be7ee7a19fb0beb91487ce311ac82ec5c7704bb9a555f2e1b", "adversarial": "95ab690a8db5eb4d52652ed70cf1c360f661685489fcf0bcd9fffb6257dd890c"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00076", "index": 76, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/167.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/167.jpg", "captions": {"source": "A container of chili with a spoon in it.", "target": "A bunch of bananas are hanging on a metal rack.", "adversarial": "A bowl of bananas with a spoon in it."}, "image_sha256": {"source": "bd903c1ba1292deed658d5eccac834c9c00b2d12f08fc6aadcb7e8b15c83b461", "target": "7e123ae300786f4f84aac181712aea932b20cf2cd70cdb84ea4d72954c02e797", "adversarial": "1dcdc32144cbe98cf8f4bb2b7bf5abdea1c00cc9e0a756b48a8519456c717d9e"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00077", "index": 77, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/168.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/168.jpg", "captions": {"source": "A large black ship with the name Apl Minnesota on the side.", "target": "A giraffe is standing on a wooden fence.", "adversarial": "A giraffe is leaning on a wooden fence."}, "image_sha256": {"source": "8aa9c595f86c7f736b7622a4148b56fb08c57698fa4e6d399192f09cbcd5eebb", "target": "04b38e6abd08db6cca2845dc1322fe2017fa254a2a1f55f53c419331ad0fdb70", "adversarial": "0baa549b9483cef9bd9116bd5916e1ac579cf0a6433ba04ed2c2de7fb4b1a73a"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00078", "index": 78, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/169.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/169.jpg", "captions": {"source": "A display of toys, including a Finding Nemo toy, sits on a shelf.", "target": "A wooden bench is sitting on a sandy beach.", "adversarial": "A bench with a blue seat and a yellow strap."}, "image_sha256": {"source": "a37be415ae9dc166bcf3488f44679f4dff0402d79383d2653f8afa95e687f7d2", "target": "1fd700668f23603c4baee569e3e19934b844e68ab5478cc6f1c1e32747f3c7db", "adversarial": "5fda330c642ab085acd798557818faf939025f64752397248faa5dad9bd6f882"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00079", "index": 79, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/17.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/17.jpg", "captions": {"source": "A cup of coffee with a heart shaped foam on top.", "target": "A young boy is holding a baseball bat and smiling.", "adversarial": "A boy wearing a baseball cap and jersey is swinging a bat."}, "image_sha256": {"source": "8ec7d3331bfe6b1bcfa1c80c4622861f19a61aec22e8df9f12f44f62236e3bc5", "target": "1e23e4785c7d321cc0efbf41b71c16c6d49d6ee3c96a2eb4fd28ce991d4a0018", "adversarial": "3eb0ba252a1aa45472b460dec45179069a04f4fb3b6c585701b8c866c7052376"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00080", "index": 80, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/170.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/170.jpg", "captions": {"source": "A bird statue is perched on a rock.", "target": "A skateboarder is performing a trick on a ramp.", "adversarial": "A skateboarder is riding a ramp in a park."}, "image_sha256": {"source": "59c2a78d50a220d90bd53a4bf126dbb8ad4d69e42e97d723ac3682462d313e90", "target": "73ee7b3093aebb8c231dd5e0802976026ed5c48015e425a988d8593c8452dda7", "adversarial": "35ddbf0805619702669a48dcd45ac336ac65e3c76d6dcf99dac51d647cb86ea4"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00081", "index": 81, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/171.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/171.jpg", "captions": {"source": "A yellow chair is placed in front of a wooden table with a red wall behind it.", "target": "A bedroom with a bed and a lamp.", "adversarial": "A bed with a yellow comforter and brown sheets."}, "image_sha256": {"source": "ad14aab41ec6579186b25e3a92d12b97aedcbf7311f40435afa04f4c0dcf2c63", "target": "68fbfe76099fdf5be9da826954af43f25c48d383a6f63bc2946ae8462f989a94", "adversarial": "58ba431d6f609c463f624a8a8630189f0201aa3a82d7c5109b311a0bded6b722"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00082", "index": 82, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/172.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/172.jpg", "captions": {"source": "A large building with a sign that says \\"Up Town\\".", "target": "A bowl of fruit with a banana, apple, and pomegranate.", "adversarial": "A wooden tray with a bowl of fruit on it."}, "image_sha256": {"source": "fde8486b8df8a19a3943d797caf7afabfce4b824dc35344776ad83801c492708", "target": "a526e6f74c1404c858736fe8d1c779ab9333f619bda1be588d6140e36903b54f", "adversarial": "8fc9391b5b3b90c5c735736ed6be47bd029dc1e2754a7d538b1537b89dda75a0"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00083", "index": 83, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/173.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/173.jpg", "captions": {"source": "A group of motorcyclists race on a track.", "target": "A brown horse with a white stripe on its face is standing in a stall.", "adversarial": "A jockey is riding a horse in a corral."}, "image_sha256": {"source": "efc5567b1310f880121456c47d4f9e9a72b5afaad5010c686e07f135b671e2af", "target": "13af0dfd9ae107ec875da7bec0ba72b77fbb83b23a128bd7b71a82b83d530369", "adversarial": "4d9262a41cfe590777aa8cb03407729d3bdabd0c33c3667f9db030b66c2ce277"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00084", "index": 84, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/174.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/174.jpg", "captions": {"source": "A bar with a large TV and many chairs.", "target": "A black dog is laying on a white blanket.", "adversarial": "A black dog is laying on a red cushion."}, "image_sha256": {"source": "86bc8b9f28a9a90bd0093c02af0585dd6cc314680ab753a4d8432507ef5edba6", "target": "9f4714497ca45b928c3685034e75231d7d15544012461a4e425f5439f0cbdb29", "adversarial": "065c7f8cf55e78b3b71eb76b7a76ca36bf257d74afa655bb91a47c3b54c046a3"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00085", "index": 85, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/175.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/175.jpg", "captions": {"source": "A man and woman are standing in front of a store display.", "target": "A man in a yellow and black hat swings a bat at a ball.", "adversarial": "A man is swinging a bat at a baseball game."}, "image_sha256": {"source": "04ba8e65c03a82be64c5b74b23fe3c09ba720f92bc4249d0b08201cef699ae70", "target": "4047f4f3ed83276a95e7cd93c242df43c3ed8195502c9fa18bec303ce9aba8fc", "adversarial": "640bcd6db6db043d54e44fe35a771933133ac4efdd73a8a55839c018e8f9a885"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00086", "index": 86, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/176.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/176.jpg", "captions": {"source": "A silver machine with a slot for coins.", "target": "A group of people are standing in front of a giraffe.", "adversarial": "A giraffe is standing in a zoo enclosure."}, "image_sha256": {"source": "c757913fbfb6b2140168b3f67e9a5e70b2292f53cf7420d9e035eadbb07c2e44", "target": "ddfa37295fcddbe875be5543eda0e3d9159f9c3addae75994ce4f8f8da354d28", "adversarial": "94f512b2997abe5a2b0b9fb6f3d8febaa3713239718cda707b1439abd4409635"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00087", "index": 87, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/177.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/177.jpg", "captions": {"source": "A person is standing in the middle of a street at night.", "target": "A cat sitting on a toilet.", "adversarial": "A cat sitting on a toilet."}, "image_sha256": {"source": "ebdb53f0d0682503648a5d450f661f96b5fac1b95b7056fd5fd7f83f0b8029b4", "target": "dacb4ba69924b0423955c57f5c945b49c270ee023f70ea0fc0613cb256c8de88", "adversarial": "1349dd2d2a517314be70b7927172b713ec0af34f198c6556b71dae9d0ad10014"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00088", "index": 88, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/178.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/178.jpg", "captions": {"source": "A carousel with a dragon on it.", "target": "A black bear is eating fruit from a tree.", "adversarial": "A black bear is eating fruit from a tree."}, "image_sha256": {"source": "b4c89b59b6f432bade8862a157bccafa933c575fb5bbbe813cd84a406cdb4a3b", "target": "d1e896a1a5610d8b43fcf53acad348c6873f8e25f65647bfd65f427a916ba4c9", "adversarial": "8554a0659fbf90047ed8150b1e02aab0133c4db5c821c9fa2efb6659134bbc59"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00089", "index": 89, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/179.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/179.jpg", "captions": {"source": "A yellow and green insect with a long antennae.", "target": "A computer keyboard and a camera are shown in a collage.", "adversarial": "A close up of a camera lens."}, "image_sha256": {"source": "04831c8ac076b2c61c83634f6a1516dd96bf9415dbe6865525279390d95a032c", "target": "91bb4019e55a38d8df1ca7b473b1c26a0d5230a992e6ed5c5b7c01f71cb66b82", "adversarial": "d6ae5bde6aa29cc8e5f2ede1bb83dbe1e4de45c1fe5acbb6c9aa19e74359df3e"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00090", "index": 90, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/18.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/18.jpg", "captions": {"source": "A barn with a red roof and a white door.", "target": "A woman is cutting a birthday cake.", "adversarial": "A birthday party is taking place in a backyard."}, "image_sha256": {"source": "b1898db52cd83459805e7abf1b498a83165b9ce01b13186fc80d5a41f0127b24", "target": "26ad86b132ec34025d575f0a97ce3922d9a66f3dd5411c9aa87f0d854cf2e5e1", "adversarial": "0403c8d80d49d2c42df2877ef433c4a03a006c997607072fcf3c498cae7dfe44"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00091", "index": 91, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/180.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/180.jpg", "captions": {"source": "An old brown car with a canvas top is on display.", "target": "A stop sign with a sign underneath it that says \\"fasten your seatbelt\\".", "adversarial": "A stop sign is attached to a metal pole."}, "image_sha256": {"source": "417140d01a74afed5df3463f35e775f8e7b5f085fac58aecfd2ab2196d9f0dbe", "target": "85b0fc5273ca5028686f8da41cda743996a652fa0bf6ff2497da7b7a79c3c7a4", "adversarial": "783eb6a593bf45944de37aa79db59935c81f5e1715763b55baf1e26388b84f25"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00092", "index": 92, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/181.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/181.jpg", "captions": {"source": "A bag of red yarn.", "target": "A row of gold fire hydrants are lined up on a sidewalk.", "adversarial": "A group of old rusted cannons are displayed in a city."}, "image_sha256": {"source": "0076510b228d63464bbbb431a36967e07c3a6a680f6ebf54521c4948c3b5c58a", "target": "b7765d8504a3f2fc0753278dc19ceb791d197b50e09da923a28cbf723e29a30f", "adversarial": "5bb09b559645491885ab49a69c986eccf5dbc4b0fe7a6efbb577b5948edc8acf"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00093", "index": 93, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/182.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/182.jpg", "captions": {"source": "A church with a red door and a white steeple.", "target": "A man is surfing on a wave.", "adversarial": "A man is surfing in front of a church."}, "image_sha256": {"source": "b7439c42400ccea5f9d668cb1fdd150aafb9ea1c493879e234c46c454b0b9c8f", "target": "3a7aad46709a5fcb88c571955da6547e32711309227c34de74f3fcc444b70180", "adversarial": "01c54963074d1c54e1307da08f89024a6aba10a1db62f2f3528b328baba2de28"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00094", "index": 94, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/183.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/183.jpg", "captions": {"source": "A group of animals grazing in a field.", "target": "A young boy is holding a donut in his hand.", "adversarial": "A young boy is laying on the ground in a field."}, "image_sha256": {"source": "41575ea8c3a4149a47b9206b63be13b7d079501f9a497644b11b1a2e4490e94c", "target": "3654c863a27dc3cffadf0e85f1561b5b98502de5e8e9cff262f6ec6b6c8b787d", "adversarial": "5d31c09e70f159d5701b4144da9ce02a135c98c6ebc43aceb4754ddc3c76da73"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00095", "index": 95, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/184.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/184.jpg", "captions": {"source": "A stone wall with a plaque on top.", "target": "A black and white photo of a clock tower.", "adversarial": "A clock tower is in the background of a city."}, "image_sha256": {"source": "412c04bd9c6f2e7628c5bffe04453a57d1c94f01022d418dd5ecde3f6c30c82a", "target": "c5297373fc1d8dd49988dff23ce3acb2ca07965c503ce78d2a0e7da8ce454f94", "adversarial": "84cb0b6f531f0b65d31c32ca60e4d8f29def42b5ca0b634d5559dbd819b0595d"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00096", "index": 96, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/185.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/185.jpg", "captions": {"source": "Two black pots with green bushes in them.", "target": "A wave crashing on the beach.", "adversarial": "A wooden fence with plants on either side."}, "image_sha256": {"source": "8005be46b1fd78c4f0894a469500987f81e74f1b0a5f785c72650f8a0bc1175e", "target": "9c120d9f8a583bd5efcfd5a9563912bc6b6b20dd044c878a543cbaefd192fadc", "adversarial": "095984b23cdf6d95bba016c1fb6bc3937e81d0329cce6ffebe79477682ab459d"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00097", "index": 97, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/186.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/186.jpg", "captions": {"source": "A man is standing on a bridge with a group of people.", "target": "A man is taking a picture of himself in a mirror.", "adversarial": "A man is taking a picture of himself in a mirror."}, "image_sha256": {"source": "938400a7b32430823bf759cbb0baefc5edf217e48c138eb41d837f07e8c20167", "target": "5da30ce55073c007ac8a6b0229fdf2963db91ba6b1de3ccc20e145ce164be8ef", "adversarial": "0832405d692e27d7b6301a5f8bcd0bfceed30d4fba8ccaf43ecef8f26a49594b"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00098", "index": 98, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/187.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/187.jpg", "captions": {"source": "A church with a cross on top of it.", "target": "A snowboarder is sitting on the snow with a snowboard attached to his feet.", "adversarial": "A snowboarder is on a snowy slope, with a sunny sky above."}, "image_sha256": {"source": "c6d94395e77b9b999baebda86f3d06ea9246b148a98852997b98b10b7efec68c", "target": "85412e11ce6fc245677ed574fe5382bf7cdea9e2e712a5abbb2f78e3198622a2", "adversarial": "cf52b951b1d0a723673f88b60e3b582fabed25919c750f09b671046fd8973624"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}, {"sample_id": "00099", "index": 99, "original_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale/nips17/188.png", "target_path": "/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/188.jpg", "captions": {"source": "Soccer players wearing red and black uniforms are standing on a field.", "target": "A street sign for Wisconsin State Fair.", "adversarial": "A group of people wearing red jackets and red hats are standing in a field."}, "image_sha256": {"source": "9e91910ee35c3491c068ae91c12bd6ecb9a7aedbe90967e14d298b8ba2b9c3db", "target": "0a3387489a644241b80ef77654395c19f717c6919f6884371e28d84d82d0be96", "adversarial": "c3b32c08d259defeb0b9aaac2770fdc6229bb3a2805bc2cf4dec4ca856a4b142"}, "caption_model": "llava-hf/llava-1.5-7b-hf", "caption_config_sha256": "0f5179964c8f33440c2b926342c224926f816c4a8677adc06b7d18580997e7b4"}]')

remember_code('embedded_captions')

V32_SOURCE_NOTEBOOK_SHA256 = 'f0e272303896bf73ec96614777b31f07f465fe0bed85b9f530ae830a53bc7e7d'


In [ ]:
# Cell 5 — Dataset discovery and preprocessing (editable research boundary).
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def count_images(path: Path) -> int:
    """Return recursive number of supported image files below ``path``."""
    return sum(p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS for p in path.rglob('*'))

def find_imagefolder(root: Path, expected_name: str) -> Path:
    """Find the largest ImageFolder named ``expected_name`` in a Kaggle dataset."""
    candidates = sorted((p for p in root.rglob(expected_name) if p.is_dir()), key=lambda p: -count_images(p))
    if not candidates: raise FileNotFoundError(f'Cannot find {expected_name} below {root}')
    return candidates[0]

def pil_to_foa_tensor(image: Image.Image) -> Tensor:
    """Convert PIL RGB image to the original FOA tensor in pixel range [0, 255]."""
    array = np.array(image, np.uint8, copy=True)
    return torch.from_numpy(array).permute(2, 0, 1).contiguous().to(torch.get_default_dtype())


# Preprocessing
def preprocess_image(input_res: int = PAPER_INPUT_RES) -> transforms.Compose:
    """Build the exact Resize → CenterCrop → RGB → pixel-tensor preprocessing of FOA."""
    return transforms.Compose([
        transforms.Resize(input_res, interpolation=InterpolationMode.BICUBIC),
        transforms.CenterCrop(input_res), transforms.Lambda(lambda x: x.convert('RGB')),
        transforms.Lambda(pil_to_foa_tensor),
    ])

class ImageFolderWithPaths(datasets.ImageFolder):
    """ImageFolder that additionally returns the source image path for manifest export."""
    def __getitem__(self, index: int):
        """Return transformed image, class label, and original filesystem path."""
        image, label = super().__getitem__(index)
        return image, label, self.samples[index][0]


def relative_dataset_path(path, anchor):
    parts = str(path).replace('\\', '/').split('/')
    positions = [i for i, part in enumerate(parts) if part == anchor]
    if len(positions) != 1 or positions[0] == len(parts) - 1:
        raise ValueError(f'Cannot identify unique dataset anchor {anchor}: {path}')
    relative = '/'.join(parts[positions[0] + 1:])
    if any(part in {'', '.', '..'} for part in relative.split('/')):
        raise ValueError('Unsafe relative dataset path.')
    return relative

def pair_key(source_path, target_path):
    return (relative_dataset_path(source_path, 'bigscale'), relative_dataset_path(target_path, 'target_images'))

def tensor_png_bytes(pixel_tensor):
    # Exactly torchvision save_image's rounding, with PNG lossless serialization.
    image = pixel_tensor.detach().cpu()
    if image.ndim == 4:
        if image.shape[0] != 1:
            raise ValueError('Only one image per checkpoint.')
        image = image[0]
    array = image.add(0.5).clamp(0, 255).permute(1, 2, 0).to(torch.uint8).numpy()
    buffer = io.BytesIO()
    Image.fromarray(array).save(buffer, format='PNG')
    return buffer.getvalue()

def png_digest_after_preprocess(path, transform):
    with Image.open(path) as image:
        pixels = transform(image)
    return hashlib.sha256(tensor_png_bytes(pixels)).hexdigest()

def load_caption_table(input_path=None):
    if input_path is None:
        input_path = CAPTION_INPUT_PATH
    if input_path is None:
        table = EMBEDDED_CAPTIONS
        origin = {**REFERENCE_PROVENANCE, 'caption_source': 'embedded'}
    else:
        path = Path(input_path)
        if path.is_dir():
            candidates = sorted(path.rglob('captions.jsonl'))
            if len(candidates) != 1:
                raise ValueError('Choose exactly one caption run or captions.jsonl.')
            path = candidates[0]
        raw = path.read_bytes()
        table = [json.loads(line) for line in raw.decode('utf-8').splitlines() if line.strip()]
        origin = {'caption_source': str(path), 'caption_input_sha256': hashlib.sha256(raw).hexdigest()}
    if not table:
        raise ValueError('Caption table is empty.')
    by_pair, seen_ids = {}, set()
    for row in table:
        sample_id = row.get('sample_id')
        if not isinstance(sample_id, str) or not sample_id or not all(c.isalnum() or c in '_-' for c in sample_id):
            raise ValueError('Caption sample_id must be a safe nonempty identifier.')
        key = pair_key(row['original_path'], row['target_path'])
        if key in by_pair or sample_id in seen_ids:
            raise ValueError(f'Duplicate caption pair/sample_id: {sample_id}')
        for role in ('source', 'target'):
            text = row.get('captions', {}).get(role)
            if not isinstance(text, str) or not text.strip():
                raise ValueError(f'Missing caption {role}: {sample_id}')
        for role in ('source', 'target'):
            digest = row.get('image_sha256', {}).get(role)
            if not isinstance(digest, str) or len(digest) != 64 or any(c not in string.hexdigits for c in digest):
                raise ValueError(f'Missing/invalid PNG checksum {role}: {sample_id}')
        by_pair[key] = row
        seen_ids.add(sample_id)
    return by_pair, origin

def resolve_concepts(row):
    sample_id = row['sample_id']
    guide = GUIDE_OVERRIDES.get(sample_id, GUIDE_TEXT_OVERRIDE)
    if guide is None:
        guide = row['captions'].get(GUIDE_CAPTION_ROLE)
    candidates = {'source': row['captions']['source'], 'target': row['captions']['target'], 'guide': guide}
    extra = list(EXTRA_CONCEPTS) + list(EXTRA_CONCEPTS_BY_SAMPLE.get(sample_id, []))
    candidates.update({f'extra_{i:02d}': text for i, text in enumerate(extra)})
    for label, text in candidates.items():
        if not isinstance(text, str) or not text.strip():
            raise ValueError(f'Empty/invalid concept {label}: {sample_id}')
    return guide, candidates

def prepare_pairs():
    if not DATASET_ROOT.is_dir():
        raise FileNotFoundError(f'Add dataset input and correct DATASET_ROOT: {DATASET_ROOT}')
    source_dir = find_imagefolder(DATASET_ROOT, 'bigscale')
    target_dir = find_imagefolder(DATASET_ROOT, 'target_images')
    source_ds = ImageFolderWithPaths(source_dir, transform=preprocess_image())
    target_ds = ImageFolderWithPaths(target_dir, transform=preprocess_image())
    table, origin = load_caption_table()
    # Match each caption pair directly, independent of ImageFolder row ordering.
    sources = {relative_dataset_path(p, 'bigscale'): (i, p) for i, (p, _) in enumerate(source_ds.samples)}
    targets = {relative_dataset_path(p, 'target_images'): p for p, _ in target_ds.samples}
    all_sample_ids = {row['sample_id'] for row in table.values()}
    for setting, overrides in [('GUIDE_OVERRIDES', GUIDE_OVERRIDES), ('EXTRA_CONCEPTS_BY_SAMPLE', EXTRA_CONCEPTS_BY_SAMPLE)]:
        unknown_ids = set(overrides) - all_sample_ids
        if unknown_ids:
            raise ValueError(f'{setting} contains unknown sample IDs: {sorted(unknown_ids)}')
    matched = []
    for key, row in table.items():
        if key[0] not in sources or key[1] not in targets:
            raise ValueError(f'Caption pair is absent from dataset: {row["sample_id"]}, {key}')
        index, source_path = sources[key[0]]
        target_path = targets[key[1]]
        guide, concepts = resolve_concepts(row)
        matched.append({'index': index, 'sample_id': row['sample_id'], 'source_path': source_path,
                        'target_path': target_path, 'pair_key': key, 'caption_record': row,
                        'guide': guide, 'concepts': concepts})
    matched.sort(key=lambda row: (row['index'], row['sample_id']))
    matched = matched[SAMPLE_START:]
    if not matched:
        raise ValueError('No caption pairs match the dataset paths; not matching by row order.')
    if NUM_SAMPLES is not None and NUM_SAMPLES > len(matched):
        raise ValueError(f'Requested {NUM_SAMPLES} pairs, only {len(matched)} have captions.')
    selected = matched if NUM_SAMPLES is None else matched[:NUM_SAMPLES]
    transform = preprocess_image()
    for pair in selected:
        for role in ('source', 'target'):
            actual = png_digest_after_preprocess(pair[f'{role}_path'], transform)
            if actual.lower() != pair['caption_record']['image_sha256'][role].lower():
                raise ValueError(f'Preprocessed PNG checksum mismatch: {pair["sample_id"]}/{role}. Check dataset/preprocessing/caption input.')
    print(f'Matched {len(matched)} caption pairs; selected {len(selected)}. Checksums verified.')
    return selected, {**origin, 'matched_pairs': len(matched)}

remember_code('data')


In [ ]:
BACKBONE_MAP = {'B16': ClipB16FeatureExtractor, 'B32': ClipB32FeatureExtractor, 'Laion': ClipLaionFeatureExtractor}

def set_foa_seed(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

class QuietEnsembleFeatureExtractor(EnsembleFeatureExtractor_ot):
    """Keep author's forward/K-means defaults; disable logging inside clustering itself."""
    def get_cluster_center(self, embedding_img, device):
        # Upstream source K-means has no iteration limit; reject NaNs before its loop.
        if embedding_img.ndim != 2 or not torch.isfinite(embedding_img).all():
            raise FloatingPointError('Non-finite/invalid source embeddings before K-means.')
        _, centers = kmeans(X=embedding_img, num_clusters=self.cluster_number,
                            distance='euclidean', device=device, tqdm_flag=False)
        if not torch.isfinite(centers).all():
            raise FloatingPointError('Non-finite source K-means centers.')
        return centers.to(device)

def build_foa_ensemble(device, backbones, cluster_number, prune_text):
    models = []
    for name in backbones:
        model = BACKBONE_MAP[name]()
        # global_local_features only accesses CLIP vision_model, never these text modules.
        # Remove before .to(device) to avoid allocating unused text weights on T4.
        if prune_text:
            for attr in ('text_model', 'text_projection'):
                if hasattr(model.model, attr):
                    delattr(model.model, attr)
        model = model.eval().requires_grad_(False).to(device)
        original_features = model.global_local_features
        use_autocast = bool(FOA_AUTOCAST)
        def precision_features(x, _forward=original_features, _autocast=use_autocast):
            with torch.autocast(device_type=x.device.type, dtype=torch.float16,
                                enabled=_autocast and x.device.type == 'cuda'):
                global_feature, local_feature = _forward(x)
            # K-means and OT must never run under autocast or consume half features.
            return global_feature.float(), local_feature.float()
        model.global_local_features = precision_features
        models.append(model)
    return QuietEnsembleFeatureExtractor(models, cluster_number=cluster_number), models


FOA_EXTRACTOR = None
FOA_MODELS = None
FOA_MODEL_SIGNATURE = None
ATTENTION_BACKEND = None

def ensure_foa_models():
    global FOA_EXTRACTOR, FOA_MODELS, FOA_MODEL_SIGNATURE
    signature = (ATTACK_DEVICE, tuple(PAPER_BACKBONES), PAPER_CLUSTER_NUMBER, PRUNE_TEXT_BRANCH, FOA_AUTOCAST)
    if FOA_EXTRACTOR is not None and signature != FOA_MODEL_SIGNATURE:
        raise RuntimeError('FOA model/device settings changed; restart session before loading another ensemble.')
    if FOA_EXTRACTOR is None:
        with preserve_rng():
            FOA_EXTRACTOR, FOA_MODELS = build_foa_ensemble(
                torch.device(ATTACK_DEVICE), PAPER_BACKBONES, PAPER_CLUSTER_NUMBER, PRUNE_TEXT_BRANCH)
            FOA_MODEL_SIGNATURE = signature
    return FOA_EXTRACTOR, FOA_MODELS

remember_code('foa_models')


In [ ]:
# Cell 6 — FOA loss. This is the paper loss copied into an editable cell.
class FOAOptimalTransportLoss(nn.Module):
    """Original FOA global/local OT loss with its dynamic ensemble weighting (K=10 by default)."""
    def __init__(self, extractors: List[nn.Module], cluster_number: int = PAPER_CLUSTER_NUMBER):
        """Initialize FOA state for the frozen surrogate extractors and local cluster count."""
        super().__init__(); self.extractors = nn.ModuleList(extractors); self.cluster_number = cluster_number
        self.ground_truth, self.ground_truth_local, self.previous_loss_list = [], [], []

    @torch.no_grad()
    def set_ground_truth(self, x: Tensor) -> None:
        """Extract target global features and K-means local centers as FOA ground truth."""
        self.ground_truth.clear(); self.ground_truth_local.clear()
        for model in self.extractors:
            global_feature, local_feature = model.global_local_features(x.to(x.device))
            centers = self.cluster_centers(local_feature.squeeze(0), x.device).unsqueeze(0)
            self.ground_truth.append(global_feature); self.ground_truth_local.append(centers)

    def cluster_centers(self, embeddings: Tensor, device: torch.device) -> Tensor:
        """Compute original Euclidean K-means local centers (100 iteration limit)."""
        if embeddings.ndim != 2 or not torch.isfinite(embeddings).all():
            raise FloatingPointError('Non-finite/invalid target embeddings before K-means.')
        _, centers = kmeans(X=embeddings, num_clusters=self.cluster_number, distance='euclidean', device=device, iter_limit=100, tqdm_flag=False)
        if not torch.isfinite(centers).all(): raise FloatingPointError('Non-finite target K-means centers.')
        return centers.to(device)

    def optimal_transport(self, source: Tensor, target: Tensor) -> Tensor:
        """Return FOA cosine-similarity optimal-transport score using original Sinkhorn settings."""
        similarity = torch.einsum('md,nd->mn', F.normalize(source, dim=1), F.normalize(target, dim=1)).contiguous()
        u = torch.full((source.shape[0],), 1 / source.shape[0], dtype=similarity.dtype, device=similarity.device)
        v = torch.full((target.shape[0],), 1 / target.shape[0], dtype=similarity.dtype, device=similarity.device)
        with torch.no_grad():
            kernel = torch.exp(-(1 - similarity) / 0.1)
            transport = self.sinkhorn(kernel, u, v)
        if not torch.isfinite(transport).all(): raise FloatingPointError('Non-finite FOA transport.')
        return torch.sum(transport * similarity)

    def sinkhorn(self, kernel: Tensor, u: Tensor, v: Tensor) -> Tensor:
        """Run the original 100-step, 1e-2-threshold Sinkhorn normalization."""
        r, c = torch.ones_like(u), torch.ones_like(v)
        for _ in range(100):
            old_r = r; r = u / (kernel @ c.unsqueeze(-1)).squeeze(-1); c = v / (kernel.T @ r.unsqueeze(-1)).squeeze(-1)
            if (r - old_r).abs().mean().item() < 1e-2: break
        return torch.outer(r, c) * kernel

    def forward(self, global_features: Dict[int, Tensor], local_features: Dict[int, Tensor]) -> Tensor:
        """Compute original FOA global OT + 0.2 local OT and dynamic model weights."""
        losses = [self.optimal_transport(self.ground_truth[i], global_features[i].unsqueeze(0)) + 0.2 * self.optimal_transport(self.ground_truth_local[i].squeeze(0), local_features[i].squeeze(0)) for i in range(len(self.extractors))]
        if not self.previous_loss_list: self.previous_loss_list = [loss.detach() for loss in losses]
        weights = np.array([loss.item() / (prev.item() + 1e-8) for loss, prev in zip(losses, self.previous_loss_list)])
        with np.errstate(over='ignore', invalid='ignore'):
            weights = np.exp(weights) / np.exp(weights).sum() * len(weights)
        if not np.isfinite(weights).all(): raise FloatingPointError('FOA dynamic weights overflowed.')
        self.previous_loss_list = [loss.detach() for loss in losses]
        return sum(float(weights[i]) * loss for i, loss in enumerate(losses))

remember_code('foa_loss')


In [ ]:
"""Fixed-area hotspot ranking with the original IoU fallback."""



def compute_box_iou(box1, box2):
    # box: (x_min, x_max, y_min, y_max) inclusive
    x1_min, x1_max, y1_min, y1_max = box1
    x2_min, x2_max, y2_min, y2_max = box2

    inter_x_min = max(x1_min, x2_min)
    inter_x_max = min(x1_max, x2_max)
    inter_y_min = max(y1_min, y2_min)
    inter_y_max = min(y1_max, y2_max)

    inter_w = max(0, inter_x_max - inter_x_min + 1)
    inter_h = max(0, inter_y_max - inter_y_min + 1)
    inter_area = inter_w * inter_h

    area1 = (x1_max - x1_min + 1) * (y1_max - y1_min + 1)
    area2 = (x2_max - x2_min + 1) * (y2_max - y2_min + 1)

    union_area = area1 + area2 - inter_area
    return inter_area / union_area if union_area > 0 else 0.0


def get_top_k_boxes_fixed_area(attention_map, target_ratio, k=1, iou_threshold=0.3, area_tolerance=0.05):
    """
    Find top-k boxes with maximum attention score that have area approximately equal to target_ratio * total_area.
    """
    H, W = attention_map.shape
    total_cells = H * W
    target_cells = int(target_ratio * total_cells)
    if target_cells < 1:
        target_cells = 1

    # When target ratio is effectively the full map, force search to stay near full area
    near_full = target_ratio >= 0.999
    area_lower = total_cells if near_full else max(1, int(target_cells * (1 - area_tolerance)))
    area_upper = total_cells if near_full else min(total_cells, int(target_cells * (1 + area_tolerance)))

    integral = np.cumsum(np.cumsum(attention_map, axis=0), axis=1)

    def get_sum(r1, c1, r2, c2):
        res = integral[r2, c2]
        if r1 > 0:
            res -= integral[r1 - 1, c2]
        if c1 > 0:
            res -= integral[r2, c1 - 1]
        if r1 > 0 and c1 > 0:
            res += integral[r1 - 1, c1 - 1]
        return res

    candidates = []

    for h in range(1, H + 1):
        w = int(round(target_cells / h))
        if w < 1:
            w = 1
        if w > W:
            w = W

        # Aspect ratio constraint to prevent elongated boxes
        if h / w > 3.0 or w / h > 3.0:
            continue

        area = h * w
        # Enforce area closeness to target to avoid undersized boxes dominating
        if area < area_lower or area > area_upper:
            continue

        for r in range(H - h + 1):
            for c in range(W - w + 1):
                current_sum = get_sum(r, c, r + h - 1, c + w - 1)
                # current_mean = current_sum / area
                candidates.append((current_sum, (c, c + w - 1, r, r + h - 1)))

    # Sort by score descending
    candidates.sort(key=lambda x: x[0], reverse=True)

    # Select top-k with IoU filtering
    selected_boxes = []
    for _, box in candidates:
        if len(selected_boxes) >= k:
            break

        # Check IoU with already selected boxes
        is_valid = True
        for selected_box in selected_boxes:
            if compute_box_iou(box, selected_box) > iou_threshold:
                is_valid = False
                break

        if is_valid:
            selected_boxes.append(box)

    # Fallback: if we could not gather k boxes under the IoU cap,
    # fill the rest with the next best candidates even if they overlap.
    if len(selected_boxes) < k and candidates:
        for _, box in candidates:
            if len(selected_boxes) >= k:
                break
            # Avoid exact duplicates
            if any(box == chosen for chosen in selected_boxes):
                continue
            selected_boxes.append(box)

    return selected_boxes


def build_schedule(attention_map, num_phases=10, top_k=3, iou_threshold=0.3):
    attention_map = np.asarray(attention_map)
    if attention_map.ndim != 2 or not attention_map.size or not np.isfinite(attention_map).all():
        raise ValueError("Attention must be a nonempty finite 2D map")
    if num_phases < 1 or top_k < 1 or not 0 <= iou_threshold <= 1:
        raise ValueError("Invalid hotspot schedule parameters")
    grid_h, grid_w = attention_map.shape
    schedule = []
    for i in range(num_phases):
        ratio = (i + 1) / num_phases
        boxes = get_top_k_boxes_fixed_area(attention_map, ratio, k=top_k, iou_threshold=iou_threshold)
        for j, (xmin, xmax, ymin, ymax) in enumerate(boxes):
            top = np.clip(ymin / grid_h, 0.0, 1.0)
            left = np.clip(xmin / grid_w, 0.0, 1.0)
            height = np.clip((ymax - ymin + 1) / grid_h, 0.0, 1.0 - top)
            width = np.clip((xmax - xmin + 1) / grid_w, 0.0, 1.0 - left)
            schedule.append(
                dict(
                    threshold=ratio,
                    box=tuple(map(float, (top, left, height, width))),
                    phase_group=i + 1,
                    sub_phase=j + 1,
                )
            )
    if not schedule:
        raise ValueError("No hotspot candidates found")
    return schedule

class RegionBiasedRandomResizedCrop(nn.Module):
    """
    RandomResizedCrop with biased sampling of hotspot region.
    """

    def __init__(
        self,
        size,
        scale=(0.8, 1.0),
        hotspot_rect: tuple[int, int, int, int] = None,
        hotspot_prob: float = 0.8,
        antialias=True,
    ):
        super().__init__()
        # Use standard RandomResizedCrop internally
        self.standard_cropper = transforms.RandomResizedCrop(size=size, scale=scale, antialias=antialias)

        if hotspot_rect is None:
            # If hotspot is not provided, always use standard crop
            self.hotspot_prob = 0.0
        else:
            self.hotspot_prob = hotspot_prob
            self.top, self.left, self.height, self.width = hotspot_rect

    def forward(self, img: torch.Tensor) -> torch.Tensor:
        # With p_hot probability, crop from the hotspot region
        if random.random() < self.hotspot_prob:
            # 1. Crop the hotspot region from the original image
            img_hotspot = TF.crop(img, self.top, self.left, self.height, self.width)

            # 2. Apply standard RandomResizedCrop to the cropped hotspot region
            #    (If the hotspot is too small to crop, an error may occur, so exception handling)
            try:
                return self.standard_cropper(img_hotspot), "hotspot"
            except ValueError:
                # If the hotspot is too small to crop, just resize
                return TF.resize(img_hotspot, self.standard_cropper.size, antialias=True), "hotspot"
        else:
            # With the remaining probability, use standard RandomResizedCrop on the entire image
            return self.standard_cropper(img), "global"


def box_to_pixels(box, height, width):
    top, left, h, w = map(float, box)
    if not all(math.isfinite(x) for x in (top, left, h, w)):
        raise ValueError('Non-finite hotspot box.')
    t, l = max(0, min(height - 1, int(top * height))), max(0, min(width - 1, int(left * width)))
    ph, pw = max(1, int(h * height)), max(1, int(w * width))
    return t, l, min(ph, height - t), min(pw, width - l)

def schedule_entry(schedule, step, steps):
    if not schedule or steps < len(schedule) or not 0 <= step < steps:
        raise ValueError('Need enough steps for every actual schedule entry.')
    block = steps // len(schedule)
    index = min(step // block, len(schedule) - 1)
    return index, schedule[index]

class TracedSourceCrop(nn.Module):
    """SAGA hotspot crop or FOA full-image crop, with exact random crop rectangle recorded."""
    def __init__(self, size, scale, hotspot_rect=None):
        super().__init__()
        self.hotspot_rect = hotspot_rect
        self.saga_cropper = RegionBiasedRandomResizedCrop(size=size, scale=scale,
                                                        hotspot_rect=hotspot_rect, hotspot_prob=1.0)
        self.last_rectangle = None

    def forward(self, image):
        # Preserve SAGA's Python-RNG draw for its hotspot branch; baseline does not draw it.
        if self.hotspot_rect is None:
            top, left, height, width = 0, 0, image.shape[-2], image.shape[-1]
        else:
            random.random()
            top, left, height, width = self.hotspot_rect
        region = TF.crop(image, top, left, height, width)
        standard = self.saga_cropper.standard_cropper
        t, l, h, w = transforms.RandomResizedCrop.get_params(region, standard.scale, standard.ratio)
        self.last_rectangle = (top + t, left + l, h, w)
        return TF.resized_crop(region, t, l, h, w, standard.size,
                               interpolation=standard.interpolation, antialias=standard.antialias)

def checkpoint_steps(steps):
    interval = max(1, steps // 10)
    return sorted({0, steps, *range(interval, steps + 1, interval)})

remember_code('saga_regions')


In [ ]:
DEBUG = os.environ.get("DEBUG", "0") == "1"

STOP_WORDS = {
    "i",
    "me",
    "my",
    "myself",
    "we",
    "our",
    "ours",
    "ourselves",
    "you",
    "you're",
    "you've",
    "you'll",
    "you'd",
    "your",
    "yours",
    "yourself",
    "yourselves",
    "he",
    "him",
    "his",
    "himself",
    "she",
    "she's",
    "her",
    "hers",
    "herself",
    "it",
    "it's",
    "its",
    "itself",
    "they",
    "them",
    "their",
    "theirs",
    "themselves",
    "what",
    "which",
    "who",
    "whom",
    "this",
    "that",
    "that'll",
    "these",
    "those",
    "am",
    "is",
    "are",
    "was",
    "were",
    "be",
    "been",
    "being",
    "have",
    "has",
    "had",
    "having",
    "do",
    "does",
    "did",
    "doing",
    "a",
    "an",
    "the",
    "and",
    "but",
    "if",
    "or",
    "because",
    "as",
    "until",
    "while",
    "of",
    "at",
    "by",
    "for",
    "with",
    "about",
    "against",
    "between",
    "into",
    "through",
    "during",
    "before",
    "after",
    "above",
    "below",
    "to",
    "from",
    "up",
    "down",
    "in",
    "out",
    "on",
    "off",
    "over",
    "under",
    "again",
    "further",
    "then",
    "once",
    "here",
    "there",
    "when",
    "where",
    "why",
    "how",
    "all",
    "any",
    "both",
    "each",
    "few",
    "more",
    "most",
    "other",
    "some",
    "such",
    "no",
    "nor",
    "not",
    "only",
    "own",
    "same",
    "so",
    "than",
    "too",
    "very",
    "s",
    "t",
    "can",
    "will",
    "just",
    "don",
    "don't",
    "should",
    "should've",
    "now",
    "d",
    "ll",
    "m",
    "o",
    "re",
    "ve",
    "y",
    "ain",
    "aren",
    "aren't",
    "couldn",
    "couldn't",
    "didn",
    "didn't",
    "doesn",
    "doesn't",
    "hadn",
    "hadn't",
    "hasn",
    "hasn't",
    "haven",
    "haven't",
    "isn",
    "isn't",
    "ma",
    "mightn",
    "mightn't",
    "mustn",
    "mustn't",
    "needn",
    "needn't",
    "shan",
    "shan't",
    "shouldn",
    "shouldn't",
    "wasn",
    "wasn't",
    "weren",
    "weren't",
    "won",
    "won't",
    "wouldn",
    "wouldn't",
}


def clean_token(token: str) -> str:
    """Lowercase, strip, and drop punctuation to normalize tokens."""
    return token.strip().lower().translate(str.maketrans("", "", string.punctuation))


def is_content_word(token: str) -> bool:
    """Return True if token is non-empty after cleaning and not a stop word."""
    cleaned = clean_token(token)
    return bool(cleaned) and cleaned not in STOP_WORDS


def filter_generated_tokens(tokenizer, token_ids: Sequence[int], use_content_words: bool) -> list[int]:
    """
    Return indices of tokens to use. If use_content_words is True, keep only content
    words; otherwise return all indices. Falls back to all tokens if filtering empties.
    """
    if not use_content_words:
        return list(range(len(token_ids)))

    logger.debug(f"[Use content words: {use_content_words}] Only keeping content words")
    valid_indices = []
    for idx, token_id in enumerate(token_ids):
        token_str = tokenizer.decode([token_id], skip_special_tokens=False)
        if is_content_word(token_str):
            valid_indices.append(idx)
        elif DEBUG:
            logger.debug(f"Filtering out token '{token_str}' at index {idx}")

    if not valid_indices:
        return list(range(len(token_ids)))

    return valid_indices


def attention_layout(input_ids, concept_start, concept_length, image_token_id, vision_start_id, vision_end_id, grid_shape):
    ids = input_ids.reshape(-1)
    image_positions = torch.nonzero(ids == image_token_id, as_tuple=False).flatten()
    starts = torch.nonzero(ids == vision_start_id, as_tuple=False).flatten()
    ends = torch.nonzero(ids == vision_end_id, as_tuple=False).flatten()
    if len(starts) != 1 or len(ends) != 1 or len(image_positions) != grid_shape[0] * grid_shape[1]:
        raise ValueError('Expected one image and matching image-token grid.')
    expected = torch.arange(int(starts[0]) + 1, int(ends[0]), device=ids.device)
    if not torch.equal(image_positions, expected):
        raise ValueError('Vision tokens must be contiguous and exclude marker tokens.')
    if concept_length < 1 or concept_start <= int(ends[0]) or concept_start + concept_length != len(ids):
        raise ValueError('Concept token interval must be the appended assistant text.')
    return image_positions

def normalize_concept_attention(token_to_vision):
    values = token_to_vision.detach().float()
    if values.ndim != 2 or not values.numel() or not torch.isfinite(values).all() or (values < 0).any():
        raise ValueError('Invalid concept-to-vision attention weights.')
    masses = values.sum(-1)
    if (masses <= 1e-12).any():
        raise ValueError('Concept has zero image-attention mass; not substituting a map.')
    probability = (values / masses[:, None]).mean(0)
    probability = probability / probability.sum()
    minimum, maximum = probability.min(), probability.max()
    minmax = (probability - minimum) / (maximum - minimum + 1e-8)
    return probability, minmax, masses


from transformers.models.qwen3_vl.modeling_qwen3_vl import apply_rotary_pos_emb, repeat_kv

class _AttentionReady(Exception):
    """Private control flow: no later layers are needed for a salient-region map."""

def selected_causal_vision_rows(module, hidden_states, position_embeddings, query_rows, vision_positions):
    """Exact eager equations for selected rows, including all-key head softmax.

    This function receives a single unpadded, uncached image/text sequence.
    Do not softmax only image keys: that would change relative head weighting.
    """
    if hidden_states.shape[0] != 1:
        raise ValueError('Attention capture requires one unpadded sequence.')
    shape = (*hidden_states.shape[:-1], -1, module.head_dim)
    q = module.q_norm(module.q_proj(hidden_states).view(shape)).transpose(1, 2)
    k = module.k_norm(module.k_proj(hidden_states).view(shape)).transpose(1, 2)
    q, k = apply_rotary_pos_emb(q, k, *position_embeddings)
    q = q.index_select(2, query_rows)
    k = repeat_kv(k, module.num_key_value_groups)
    scores = torch.matmul(q, k.transpose(2, 3)) * module.scaling
    future = torch.arange(k.shape[2], device=k.device)[None, :] > query_rows[:, None]
    scores = scores.masked_fill(future[None, None], torch.finfo(scores.dtype).min)
    weights = F.softmax(scores, dim=-1, dtype=torch.float32).to(q.dtype)
    return weights[0].index_select(2, vision_positions).float().mean(0)

class ConceptAttentionExtractor:
    """SDPA prefix, selected Q/K rows at layer 29, early exit, per-image vision cache."""
    def __init__(self, resolved_revision=None):
        self.device = torch.device(ATTENTION_DEVICE)
        self.signature = self.current_signature()
        requested_revision = resolved_revision or ATTENTION_REVISION
        resolved_config = AutoConfig.from_pretrained(ATTENTION_MODEL_ID, revision=requested_revision)
        load_revision = getattr(resolved_config, '_commit_hash', None) or requested_revision
        self.processor = AutoProcessor.from_pretrained(ATTENTION_MODEL_ID, revision=load_revision)
        quantization = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
            bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16,
            llm_int8_skip_modules=['visual'])
        self.model = AutoModelForImageTextToText.from_pretrained(
            ATTENTION_MODEL_ID, revision=load_revision, torch_dtype=torch.float16,
            quantization_config=quantization, device_map={'': self.device.index},
            attn_implementation='sdpa', low_cpu_mem_usage=True, trust_remote_code=False).eval()
        self.model.requires_grad_(False)
        layers = self.model.model.language_model.layers
        if not 0 <= ATTENTION_LAYER < len(layers):
            raise ValueError('Attention layer index is unavailable.')
        self.attention_module = layers[ATTENTION_LAYER].self_attn
        self._lock = threading.RLock()
        self._prepared = None
        self.metadata = {'model_id': ATTENTION_MODEL_ID,
            'resolved_revision': getattr(self.model.config, '_commit_hash', None),
            'layer': ATTENTION_LAYER, 'dtype': 'float16', 'quantization': 'nf4_double',
            'device': str(self.device), 'input_res': ATTENTION_INPUT_RES,
            'mode': 'teacher_forced_concept_input_rows', 'attn_implementation': 'sdpa_prefix_selected_eager_rows',
            'early_exit': 'before selected self-attention value/output projection and MLP',
            'vision_cache_scope': 'same checkpoint image only', 'use_cache': False}

    @staticmethod
    def current_signature():
        return (ATTENTION_MODEL_ID, ATTENTION_REVISION, ATTENTION_DEVICE, ATTENTION_LAYER,
                ATTENTION_INPUT_RES, ATTENTION_PROMPT, MAX_CONCEPT_TOKENS, CONTENT_WORDS_ONLY)

    @contextlib.contextmanager
    def image_context(self, image):
        # One thread owns model hooks, the processor and its mutable RoPE state.
        device_context = torch.cuda.device(self.device) if self.device.type == 'cuda' else contextlib.nullcontext()
        with self._lock, torch.inference_mode(), device_context:
            if self.current_signature() != self.signature:
                raise RuntimeError('Attention settings changed during a run; concept protocol must stay fixed.')
            if self.model.training:
                raise RuntimeError('Attention must run in eval mode without dropout/RNG draws.')
            if self._prepared is not None:
                raise RuntimeError('Nested image contexts are not supported.')
            image = image.convert('RGB').resize((ATTENTION_INPUT_RES, ATTENTION_INPUT_RES), Image.Resampling.BICUBIC)
            messages = [{'role': 'user', 'content': [{'type': 'image'}, {'type': 'text', 'text': ATTENTION_PROMPT}]}]
            prompt = self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            base = self.processor(text=[prompt], images=[image], do_resize=False, return_tensors='pt')
            self._prepared = {key: value.to(self.device) for key, value in base.items()}
            visual = self.model.model.visual
            original_forward = visual.forward
            had_override = 'forward' in visual.__dict__
            cache = {}
            def cached_visual(*args, **kwargs):
                if 'features' not in cache:
                    cache['features'] = original_forward(*args, **kwargs)
                return cache['features']
            visual.forward = cached_visual
            try:
                yield self
            finally:
                if had_override:
                    visual.forward = original_forward
                else:
                    del visual.forward
                self._prepared = None
                cache.clear()
                # An early exit can leave RoPE deltas set; every concept computes fresh positions.
                self.model.model.rope_deltas = None

    @torch.inference_mode()
    def extract_prepared(self, concept):
        if self.current_signature() != self.signature:
            raise RuntimeError('Attention settings changed during a run; restart with a fixed configuration.')
        if self._prepared is None:
            raise RuntimeError('Use image_context before extract_prepared.')
        if not isinstance(concept, str) or not concept.strip():
            raise ValueError('Concept must be nonempty text.')
        inputs = dict(self._prepared)
        tokenizer = self.processor.tokenizer
        concept_ids = tokenizer.encode(concept, add_special_tokens=False)
        if not concept_ids or len(concept_ids) > MAX_CONCEPT_TOKENS:
            raise ValueError(f'Concept needs 1..{MAX_CONCEPT_TOKENS} tokens; no silent truncation.')
        if any(token in tokenizer.all_special_ids for token in concept_ids):
            raise ValueError('Concept must not contain special control tokens.')
        start = inputs['input_ids'].shape[1]
        suffix = torch.tensor([concept_ids], device=self.device, dtype=inputs['input_ids'].dtype)
        inputs['input_ids'] = torch.cat((inputs['input_ids'], suffix), dim=1)
        inputs['attention_mask'] = torch.ones_like(inputs['input_ids'])
        inputs.pop('position_ids', None); inputs.pop('token_type_ids', None)
        grid = inputs['image_grid_thw'][0].cpu()
        merge = self.processor.image_processor.merge_size
        if int(grid[0]) != 1 or int(grid[1]) % merge or int(grid[2]) % merge:
            raise ValueError('Expected one still-image divisible merged grid.')
        shape = (int(grid[1]) // merge, int(grid[2]) // merge)
        config = self.model.config
        vision = attention_layout(inputs['input_ids'][0], start, len(concept_ids),
            config.image_token_id, config.vision_start_token_id, config.vision_end_token_id, shape)
        offsets = filter_generated_tokens(tokenizer, concept_ids, CONTENT_WORDS_ONLY)
        queries = torch.tensor([start + offset for offset in offsets], device=self.device)
        captured = {}
        def capture(module, args, kwargs):
            if kwargs.get('past_key_values') is not None:
                raise RuntimeError('KV cache is forbidden during attention capture.')
            hidden = kwargs.get('hidden_states', args[0] if args else None)
            position_embeddings = kwargs.get('position_embeddings')
            if hidden is None or position_embeddings is None:
                raise RuntimeError('Unexpected Qwen self-attention signature; install pinned Transformers.')
            captured['weights'] = selected_causal_vision_rows(module, hidden, position_embeddings, queries, vision).cpu()
            raise _AttentionReady()
        handle = self.attention_module.register_forward_pre_hook(capture, with_kwargs=True)
        self.model.model.rope_deltas = None
        try:
            try:
                self.model.model(**inputs, use_cache=False, output_attentions=False,
                                 output_hidden_states=False, return_dict=True)
            except _AttentionReady:
                pass
        finally:
            handle.remove()
        if 'weights' not in captured:
            raise RuntimeError('Selected attention layer was not reached.')
        probability, minmax, masses = normalize_concept_attention(captured['weights'])
        return {'probability': probability.reshape(shape).numpy(), 'minmax': minmax.reshape(shape).numpy(),
            'metadata': {'concept': concept, 'token_ids': concept_ids, 'selected_offsets': offsets,
                'selected_token_texts': [tokenizer.decode([concept_ids[i]]) for i in offsets],
                'concept_query_rows': queries.cpu().tolist(), 'vision_positions': vision.cpu().tolist(),
                'grid_shape': list(shape), 'mean_raw_vision_mass': float(masses.mean()),
                'method': 'teacher_forced_concept_input_rows', 'layer': ATTENTION_LAYER,
                'backend': 'sdpa_prefix_selected_eager_rows', 'early_exit': True}}

    def extract(self, image, concept):
        with self.image_context(image):
            return self.extract_prepared(concept)

def ensure_attention_backend(resolved_revision=None):
    global ATTENTION_BACKEND
    if ATTENTION_BACKEND is not None and ConceptAttentionExtractor.current_signature() != ATTENTION_BACKEND.signature:
        raise RuntimeError('Qwen configuration changed; restart session before reloading.')
    if (ATTENTION_BACKEND is not None and resolved_revision is not None
            and ATTENTION_BACKEND.metadata.get('resolved_revision') != resolved_revision):
        raise RuntimeError('Cached Qwen revision differs from the resumed run; restart the session.')
    if ATTENTION_BACKEND is None:
        with preserve_rng():
            ATTENTION_BACKEND = ConceptAttentionExtractor(resolved_revision=resolved_revision)
    return ATTENTION_BACKEND

remember_code('concept_attention')


In [ ]:
def probability_overlay(image, probability, axis, title, vmax, rectangle=None):
    axis.imshow(image)
    axis.imshow(probability, extent=(0, image.width, image.height, 0), interpolation='bilinear',
                cmap='inferno', vmin=0, vmax=vmax, alpha=0.55)
    if rectangle is not None:
        top, left, height, width = rectangle
        axis.add_patch(Rectangle((left, top), width, height, fill=False, edgecolor='cyan', linewidth=1.4))
    axis.set_title('\n'.join(textwrap.fill(line, width=42) for line in title.splitlines()), fontsize=9)
    axis.axis('off')

def preview_checkpoint(image_path, results, step, hotspot=None, crop=None, handle=None, scales=None):
    image = Image.open(image_path).convert('RGB')
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(image); axes[0].set_title(f'Step {step}: adversarial'); axes[0].axis('off')
    for rect, color in ((hotspot, 'cyan'), (crop, 'lime')):
        if rect is not None:
            t, l, h, w = rect
            axes[0].add_patch(Rectangle((l, t), w, h, fill=False, edgecolor=color, linewidth=1.3))
    for axis, label in zip(axes[1:], ('source', 'target')):
        result = results[label]
        # Initial-image maximum is frozen for live playback; final timeline uses all-step maximum.
        vmax = scales[label] if scales is not None else 1.0
        probability_overlay(image, result['probability'], axis,
                            f'{label}: {result["metadata"]["concept"]}\nFixed scale 0..{vmax:.4g}', vmax, hotspot)
    fig.tight_layout()
    if handle is None:
        handle = display(fig, display_id=True)
    else:
        handle.update(fig)
    plt.close(fig)
    return handle

def plot_attention_timeline(sample_dir, attention_root=None, figure_dir=None, show=True):
    sample_dir = Path(sample_dir)
    attention_root = sample_dir / 'attention' if attention_root is None else Path(attention_root)
    figure_dir = sample_dir / 'figures' if figure_dir is None else Path(figure_dir)
    figure_dir.mkdir(parents=True, exist_ok=True)
    metadata_path = attention_root / 'concepts.json'
    if not metadata_path.is_file():
        return []
    concepts = json.loads(metadata_path.read_text(encoding='utf-8'))
    geometry_path = sample_dir / 'attention' / 'checkpoint_metrics.jsonl'
    geometry = {}
    if geometry_path.is_file():
        geometry = {row['step']: row for row in
                    (json.loads(line) for line in geometry_path.read_text(encoding='utf-8').splitlines() if line.strip())}
    outputs = []
    for label, concept in concepts.items():
        files = sorted(attention_root.glob(f'step_*/{label}.npz'))
        if not files:
            continue
        steps = [int(path.parent.name.split('_')[-1]) for path in files]
        maps = [np.load(path)['probability'] for path in files]
        vmax = max(float(values.max()) for values in maps)
        limit = max(float(np.abs(values - maps[0]).max()) for values in maps) or 1e-12
        fig, axes = plt.subplots(2, len(files), figsize=(max(6, 2.3 * len(files)), 5), squeeze=False)
        for column, (step, probability) in enumerate(zip(steps, maps)):
            with Image.open(sample_dir / 'checkpoints' / f'step_{step:05d}.png') as img:
                image = img.convert('RGB')
            region = geometry.get(step, {})
            hotspot = region.get('hotspot')
            if step == 0 and region.get('initial_hotspot_box') is not None:
                hotspot = box_to_pixels(region['initial_hotspot_box'], image.height, image.width)
            probability_overlay(image, probability, axes[0, column], f'Step {step}', vmax, hotspot)
            if region.get('last_crop') is not None:
                t, l, h, w = region['last_crop']
                axes[0, column].add_patch(Rectangle((l, t), w, h, fill=False, edgecolor='lime', linewidth=.9))
            difference = axes[1, column].imshow(probability - maps[0], cmap='coolwarm', vmin=-limit, vmax=limit)
            axes[1, column].set_title('Δ probability vs step 0', fontsize=8); axes[1, column].axis('off')
        fig.suptitle(f'{label}: {concept}\nShared probability scale: 0..{vmax:.5g}')
        fig.colorbar(axes[0, -1].images[1], ax=list(axes[0]), shrink=0.65, label='Attention probability')
        fig.colorbar(difference, ax=list(axes[1]), shrink=0.65, label='Δ probability')
        path = figure_dir / f'timeline_{label}.png'
        fig.savefig(path, dpi=140, bbox_inches='tight'); outputs.append(str(path))
        if show:
            display(fig)
        plt.close(fig)
    trace_path = sample_dir / 'trace.csv'
    checkpoints_path = attention_root / 'checkpoint_metrics.jsonl'
    if trace_path.is_file() and checkpoints_path.is_file():
        with trace_path.open(encoding='utf-8', newline='') as stream:
            trace = list(csv.DictReader(stream))
        records = [json.loads(line) for line in checkpoints_path.read_text(encoding='utf-8').splitlines() if line.strip()]
        fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
        if trace:
            axes[0].plot([int(r['step']) for r in trace], [float(r['objective']) for r in trace])
            axes[2].plot([int(r['step']) for r in trace], [float(r['linf_internal_255']) for r in trace], label='Internal float')
        for label in concepts:
            values = [r for r in records if label in r['concept_metrics']]
            if values:
                axes[1].plot([r['step'] for r in values],
                             [r['concept_metrics'][label]['initial_hotspot_mass'] for r in values], label=label)
        axes[2].plot([r['step'] for r in records], [r['linf_saved_255'] for r in records], 'o', label='PNG')
        for axis, title in zip(axes, ('FOA objective (maximize)', 'Mass in fixed initial guide hotspot', 'L∞ perturbation (0–255)')):
            axis.set_title(title); axis.set_xlabel('Step'); axis.grid(alpha=.25)
        axes[1].legend(); axes[2].legend(); fig.tight_layout()
        path = figure_dir / 'metrics.png'; fig.savefig(path, dpi=140); outputs.append(str(path))
        if show:
            display(fig)
        plt.close(fig)
    return outputs

remember_code('visualization')


In [ ]:
def hotspot_grid_mass(probability, box):
    height, width = probability.shape
    top, left, h, w = box_to_pixels(box, height, width)
    return float(probability[top:top + h, left:left + w].sum())

class CheckpointObserver:
    """Fresh guide barrier + bounded background observation on GPU 1.

    No preserve_rng in a background thread: restoring global RNG there would race
    with attack sampling. The inference-only worker performs no random operations.
    """
    def __init__(self, sample_dir, concepts, backend, *, live=False, initial_box=None, total_steps=STEPS, epsilon=PAPER_EPSILON):
        self.sample_dir = Path(sample_dir)
        self._fixed_concepts = dict(concepts)
        if 'guide' not in self._fixed_concepts:
            raise ValueError('A fixed guide concept is required.')
        self.concept_hashes = {label: concept_sha256(text) for label, text in self._fixed_concepts.items()}
        self.concepts = MappingProxyType(self._fixed_concepts.copy())
        self.token_signatures = {}
        self.observe_every = OBSERVE_ALL_CONCEPTS_EVERY
        self.epsilon = epsilon
        self.backend, self.live, self.initial_box = backend, live, initial_box
        self.total_steps = total_steps
        self.snapshot_steps = checkpoint_steps(total_steps)
        self.saved_steps, self.finished_steps = set(), set()
        self.initial_probabilities, self.live_scales = {}, {}
        self.preview_handle = None
        self.attention_seconds = self.guide_wait_seconds = 0.0
        self.pending = None
        self.closed = False
        self.pool = ThreadPoolExecutor(max_workers=1, thread_name_prefix='qwen_attention') if ASYNC_OBSERVATION else None
        (self.sample_dir / 'checkpoints').mkdir(parents=True, exist_ok=True)
        (self.sample_dir / 'attention').mkdir(parents=True, exist_ok=True)
        write_json(self.sample_dir / 'attention' / 'concepts.json', dict(self.concepts))
        write_json(self.sample_dir / 'attention' / 'concept_fingerprints.json', self.concept_hashes)

    def validate_result(self, label, result):
        text = self._fixed_concepts[label]
        metadata = dict(result['metadata'])
        if metadata.get('concept') != text:
            raise ValueError(f'Backend used a different concept for {label}.')
        probability, minmax = np.asarray(result['probability']), np.asarray(result['minmax'])
        if (probability.ndim != 2 or not probability.size or probability.shape != minmax.shape
                or not np.isfinite(probability).all() or probability.min() < 0
                or not np.isclose(probability.sum(), 1, atol=1e-5)
                or not np.isfinite(minmax).all() or minmax.min() < 0 or minmax.max() > 1 + 1e-5):
            raise ValueError('Invalid attention maps; refusing to guide an attack with them.')
        raw_mass = metadata.get('mean_raw_vision_mass', 0)
        if not math.isfinite(raw_mass) or raw_mass <= 0:
            raise ValueError('Invalid raw image-attention mass.')
        required = ('token_ids', 'selected_offsets', 'concept_query_rows', 'vision_positions', 'grid_shape', 'method', 'layer')
        if any(key not in metadata for key in required):
            raise ValueError('Attention metadata must prove concept-token/image-token alignment.')
        ids, offsets = metadata['token_ids'], metadata['selected_offsets']
        rows, vision = metadata['concept_query_rows'], metadata['vision_positions']
        if (not ids or not offsets or len(rows) != len(offsets) or sorted(set(offsets)) != offsets
                or any(type(i) is not int or not 0 <= i < len(ids) for i in offsets)
                or len(vision) != probability.size or tuple(metadata['grid_shape']) != probability.shape
                or not vision or min(rows) <= max(vision)
                or len(set(row - offset for row, offset in zip(rows, offsets))) != 1):
            raise ValueError('Invalid concept-query or image-key token alignment.')
        signature = attention_token_fingerprint(metadata)
        if label in self.token_signatures and self.token_signatures[label] != signature:
            raise ValueError(f'Concept tokens/grid changed between checkpoints: {label}.')
        self.token_signatures[label] = signature
        metadata.update(concept_sha256=self.concept_hashes[label], token_signature_sha256=signature)
        return {**result, 'metadata': metadata}

    def labels_at(self, step):
        ordinal = self.snapshot_steps.index(step)
        all_labels = step in (0, self.total_steps) or (
            self.observe_every > 0 and ordinal % self.observe_every == 0)
        return ['guide'] + [label for label in self.concepts if label != 'guide' and all_labels]

    def _work(self, path, step, clean_array, hotspot, crop, guide_future):
        started = time.perf_counter()
        try:
            with Image.open(path) as image:
                image = image.convert('RGB')
            linf = int(np.abs(np.asarray(image).astype(np.int16) - clean_array).max())
            if linf > math.ceil(self.epsilon):
                raise AssertionError('Checkpoint PNG exceeds perturbation budget.')
            output_dir = self.sample_dir / 'attention' / f'step_{step:05d}'
            output_dir.mkdir(exist_ok=False)
            results, by_text = {}, {}
            manager = self.backend.image_context(image) if hasattr(self.backend, 'image_context') else contextlib.nullcontext()
            with manager:
                for label in self.labels_at(step):
                    text = self.concepts[label]
                    if text not in by_text:
                        by_text[text] = (self.backend.extract_prepared(text) if hasattr(self.backend, 'extract_prepared')
                                         else self.backend.extract(image, text))
                    result = self.validate_result(label, by_text[text])
                    probability = np.asarray(result['probability'])
                    if not np.isfinite(probability).all() or probability.min() < 0 or not np.isclose(probability.sum(), 1, atol=1e-5):
                        raise ValueError('Backend returned an invalid probability map.')
                    results[label] = result
                    np.savez_compressed(output_dir / f'{label}.npz', probability=probability, minmax=result['minmax'])
                    write_json(output_dir / f'{label}.json', {**result['metadata'], 'checkpoint_sha256': sha256_file(path),
                        'map_sha256': sha256_file(output_dir / f'{label}.npz')})
                    if label == 'guide':
                        if self.initial_box is None:
                            self.initial_box = build_schedule(result['minmax'], NUM_PHASES, HOTSPOT_TOP_K, HOTSPOT_IOU)[0]['box']
                        # A fresh saved-PNG map is ready. GPU 0 may start its next update now.
                        guide_future.set_result({'guide': result})
            metrics = {}
            for label, result in results.items():
                probability = result['probability']
                self.initial_probabilities.setdefault(label, probability.copy())
                self.live_scales.setdefault(label, float(probability.max()))
                initial = self.initial_probabilities[label]
                if initial.shape != probability.shape:
                    raise ValueError('Attention grid changed between checkpoints.')
                metrics[label] = {'initial_hotspot_mass': hotspot_grid_mass(probability, self.initial_box),
                    'l1_vs_initial': float(np.abs(probability - initial).sum()),
                    'mean_raw_vision_mass': result['metadata']['mean_raw_vision_mass']}
            record = {'step': step, 'linf_saved_255': linf, 'hotspot': hotspot, 'last_crop': crop,
                'initial_hotspot_box': self.initial_box, 'concept_metrics': metrics,
                'observed_labels': list(results), 'checkpoint_sha256': sha256_file(path),
                'guide_concept_sha256': self.concept_hashes['guide'],
                'guide_token_signature_sha256': self.token_signatures['guide']}
            with (self.sample_dir / 'attention' / 'checkpoint_metrics.jsonl').open('a', encoding='utf-8') as stream:
                stream.write(json.dumps(record, allow_nan=False) + '\n')
            self.finished_steps.add(step)
            return results
        except BaseException as error:
            if not guide_future.done():
                guide_future.set_exception(error)
            raise
        finally:
            self.attention_seconds += time.perf_counter() - started

    def drain(self):
        if self.pending is not None:
            future, path, step, hotspot, crop = self.pending
            self.pending = None
            results = future.result()  # Propagate auxiliary failures; never silently discard.
            if self.live and 'source' in results and 'target' in results:
                self.preview_handle = preview_checkpoint(path, results, step, hotspot, crop,
                    self.preview_handle, self.live_scales)

    def capture(self, step, pixels, clean_pixels, *, hotspot=None, crop=None):
        if dict(self.concepts) != self._fixed_concepts:
            raise ValueError('Concept text changed inside a sample; start a new run instead.')
        if self.closed or step in self.saved_steps or step not in self.snapshot_steps:
            raise ValueError('Closed observer, duplicate or unexpected checkpoint.')
        self.drain()  # Bounded queue: at most one checkpoint owns Qwen memory.
        path = self.sample_dir / 'checkpoints' / f'step_{step:05d}.png'
        path.write_bytes(tensor_png_bytes(pixels))
        if not hasattr(self, 'clean_array'):
            self.clean_array = np.asarray(Image.open(io.BytesIO(tensor_png_bytes(clean_pixels)))).astype(np.int16)
        guide_future = Future()
        arguments = (path, step, self.clean_array, hotspot, crop, guide_future)
        started = time.perf_counter()
        if self.pool is None:
            future = Future()
            try:
                future.set_result(self._work(*arguments))
            except BaseException as error:
                future.set_exception(error)
        else:
            future = self.pool.submit(self._work, *arguments)
        self.pending = (future, path, step, hotspot, crop)
        results = guide_future.result()
        self.guide_wait_seconds += time.perf_counter() - started
        self.saved_steps.add(step)
        return results

    def close(self):
        if self.closed:
            return
        self.closed = True
        try:
            self.drain()
        finally:
            if self.pool is not None:
                self.pool.shutdown(wait=True, cancel_futures=False)

remember_code('observer')


In [ ]:
def foa_attention_attack(image_org, image_tgt, extractor, loss_fn, *, observer=None,
                         sample_dir=None, mode='dynamic', steps=STEPS,
                         epsilon=PAPER_EPSILON, alpha=PAPER_ALPHA, on_step=None,
                         source_crop=None, target_crop=None, on_phase=None):
    if mode not in {'baseline', 'fixed', 'dynamic'}:
        raise ValueError('Unknown attack mode.')
    if image_org.ndim != 4 or image_org.shape[0] != 1 or image_org.shape != image_tgt.shape:
        raise ValueError('FOA supports one equally sized source/target image pair.')
    if not torch.isfinite(image_org).all() or not torch.isfinite(image_tgt).all():
        raise ValueError('Non-finite input image.')
    if source_crop is None:
        source_crop = TracedSourceCrop(PAPER_INPUT_RES, PAPER_CROP_SCALE)
    if target_crop is None:
        target_crop = transforms.RandomResizedCrop(PAPER_INPUT_RES, scale=PAPER_CROP_SCALE)
    if mode != 'baseline' and observer is None:
        raise ValueError('Attention-guided attack needs a checkpoint observer.')
    if observer is not None and (observer.total_steps != steps or observer.epsilon != epsilon):
        raise ValueError('Attack and observer must use the same steps/perturbation budget.')
    sample_dir = Path(sample_dir) if sample_dir is not None else None
    delta = torch.zeros_like(image_org, requires_grad=True)
    schedule, schedule_version = None, 0
    snapshots = set(checkpoint_steps(steps))
    trace_rows = []
    trace_handle = None
    if sample_dir is not None:
        sample_dir.mkdir(parents=True, exist_ok=True)
        trace_handle = (sample_dir / 'trace.csv').open('w', encoding='utf-8', newline='')
        trace_writer = csv.DictWriter(trace_handle, fieldnames=['step', 'objective', 'linf_internal_255',
            'schedule_version', 'schedule_index', 'phase_group', 'sub_phase', 'hotspot', 'crop', 'seconds'])
        trace_writer.writeheader(); trace_handle.flush()

    def update_schedule(results, completed_steps):
        nonlocal schedule, schedule_version
        schedule = build_schedule(results['guide']['minmax'], NUM_PHASES, HOTSPOT_TOP_K, HOTSPOT_IOU)
        if steps < len(schedule):
            raise ValueError(f'STEPS={steps} must visit {len(schedule)} schedule entries.')
        schedule_version = completed_steps
        if sample_dir is not None:
            write_json(sample_dir / 'schedules' / f'step_{completed_steps:05d}.json',
                       { 'computed_at_step': completed_steps, 'applies_from_update': completed_steps + 1, 'guide': observer.concepts['guide'],
                        'guide_concept_sha256': results['guide']['metadata']['concept_sha256'],
                        'guide_token_signature_sha256': results['guide']['metadata']['token_signature_sha256'],
                        'checkpoint_sha256': sha256_file(observer.sample_dir / 'checkpoints' / f'step_{completed_steps:05d}.png'),
                        'schedule': schedule, 'steps_per_item': steps // len(schedule),
                        'selection': 'absolute_step // (total_steps // actual_schedule_length); last receives remainder'})

    try:
        if observer is not None:
            if on_phase is not None: on_phase('attention', 0)
            results = observer.capture(0, image_org, image_org)
            if mode != 'baseline':
                update_schedule(results, 0)
            if on_phase is not None: on_phase('attack', 0)
        for step in range(steps):
            started = time.perf_counter()
            if mode == 'baseline':
                cropper, hotspot, schedule_index, entry = source_crop, None, -1, {}
            else:
                schedule_index, entry = schedule_entry(schedule, step, steps)
                hotspot = box_to_pixels(entry['box'], image_org.shape[-2], image_org.shape[-1])
                cropper = TracedSourceCrop(PAPER_INPUT_RES, PAPER_CROP_SCALE, hotspot)
            with torch.no_grad():
                loss_fn.set_ground_truth(target_crop(image_tgt))
            adv_image = image_org + delta
            # Full pass still runs first and advances FOA dynamic weights.
            # Grad graph is needed only when identity crop selects this full score.
            identity = isinstance(cropper, nn.Identity)
            with contextlib.nullcontext() if identity else torch.no_grad():
                global_features, local_features = extractor(adv_image)
                global_score = loss_fn(global_features, local_features)
            if identity:
                objective = global_score
            else:
                del global_features, local_features, global_score
                crop_global, crop_local = extractor(cropper(adv_image))
                objective = loss_fn(crop_global, crop_local)
            if not torch.isfinite(objective).all():
                raise FloatingPointError('Non-finite FOA objective.')
            gradient = torch.autograd.grad(objective * GRADIENT_SCALE, delta, create_graph=False)[0]
            if not torch.isfinite(gradient).all():
                raise FloatingPointError('Non-finite FOA gradient.')
            with torch.no_grad():
                delta.copy_((delta + alpha * gradient.sign()).clamp(-epsilon, epsilon))
            rectangle = getattr(cropper, 'last_rectangle', None)
            row = {'step': step + 1, 'objective': float(objective.detach()),
                   'linf_internal_255': float(delta.detach().abs().max()),
                   'schedule_version': schedule_version, 'schedule_index': schedule_index,
                   'phase_group': entry.get('phase_group', 0), 'sub_phase': entry.get('sub_phase', 0),
                   'hotspot': json.dumps(hotspot), 'crop': json.dumps(rectangle),
                   'seconds': time.perf_counter() - started}
            trace_rows.append(row)
            if trace_handle is not None:
                trace_writer.writerow(row)
            if on_step is not None and ((step + 1) % PROGRESS_INTERVAL == 0 or step + 1 == steps):
                on_step(step + 1)
            # Release full/crop forward graphs BEFORE Qwen checkpoint inference.
            del objective, gradient, adv_image
            if not identity:
                del crop_global, crop_local
            else:
                del global_features, local_features, global_score
            completed = step + 1
            if trace_handle is not None and completed in snapshots:
                trace_handle.flush()
            if observer is not None and completed in snapshots:
                if on_phase is not None: on_phase('attention', completed)
                results = observer.capture(completed, image_org + delta.detach(), image_org,
                                           hotspot=hotspot, crop=rectangle)
                if mode == 'dynamic' and completed < steps:
                    update_schedule(results, completed)
                if on_phase is not None: on_phase('attack' if completed < steps else 'finalizing_attention', completed)
        if float(delta.detach().abs().max()) > epsilon + 1e-5:
            raise AssertionError('Internal perturbation budget violated.')
        return ((image_org + delta.detach()) / 255.0).clamp(0, 1), trace_rows
    finally:
        try:
            if observer is not None:
                observer.close()
        finally:
            if trace_handle is not None:
                trace_handle.close()

remember_code('attack')


In [ ]:
def verify_baseline_parity():
    """CPU checks: same full/crop loss order and update as v1.2, RNG-neutral observers."""
    class MockExtractor(nn.Module):
        def forward(self, x):
            return {0: x.mean((2, 3)).squeeze(0)}, {0: x.mean((2, 3)).unsqueeze(1)}
    class MockLoss(nn.Module):
        def __init__(self):
            super().__init__(); self.calls = []
        def set_ground_truth(self, x):
            self.target = x.mean((2, 3)).detach(); self.calls.append('target')
        def forward(self, global_f, local_f):
            self.calls.append('loss')
            return -(global_f[0] - self.target.squeeze(0)).square().mean()
    source = (REPO_DIR / 'generate_adversarial_samples_foa_attack.py').read_text(encoding='utf-8')
    original_node = next(node for node in ast.parse(source).body if isinstance(node, ast.FunctionDef) and node.name == 'fgsm_attack')
    namespace = {'torch': torch, 'nn': nn, 'tqdm': lambda seq, **kw: seq,
                 'log_metrics': lambda *a, **kw: None, 'MainConfig': object, 'Optional': Optional, 'transforms': transforms}
    exec(compile(ast.Module(body=[original_node], type_ignores=[]), '<original-foa>', 'exec'), namespace)
    from types import SimpleNamespace
    x, y = torch.full((1, 3, 16, 16), 100.0), torch.full((1, 3, 16, 16), 130.0)
    for cropped in (False, True):
        def crop():
            return transforms.RandomResizedCrop(16, scale=PAPER_CROP_SCALE) if cropped else nn.Identity()
        config = SimpleNamespace(optim=SimpleNamespace(steps=3, epsilon=16, alpha=1),
                                 model=SimpleNamespace(use_source_crop=cropped))
        set_foa_seed(123)
        reference_loss = MockLoss()
        reference = namespace['fgsm_attack'](config, MockExtractor(), reference_loss, crop(), crop(), 0, x, y)
        set_foa_seed(123)
        ours_loss = MockLoss()
        ours, _ = foa_attention_attack(x, y, MockExtractor(), ours_loss, mode='baseline', steps=3,
                                      source_crop=crop(), target_crop=crop())
        assert torch.equal(reference, ours), 'Baseline changed FOA update.'
        assert reference_loss.calls == ours_loss.calls, 'Baseline changed FOA full/crop ordering.'
    set_foa_seed(42)
    with preserve_rng():
        random.random(); np.random.rand(); torch.rand(10)
    observed = (random.random(), np.random.rand(), torch.rand(3))
    set_foa_seed(42)
    expected = (random.random(), np.random.rand(), torch.rand(3))
    assert observed[:2] == expected[:2] and torch.equal(observed[2], expected[2])
    assert checkpoint_steps(300) == list(range(0, 301, 30))
    print('Baseline crop on/off, loss order, checkpoint schedule and RNG checks passed.')

if RUN_PARITY_CHECK:
    with preserve_rng():
        verify_baseline_parity()

remember_code('parity')


In [ ]:
def current_run_config():
    names = ['NUM_SAMPLES', 'STEPS', 'PAPER_EPSILON', 'PAPER_ALPHA', 'PAPER_INPUT_RES', 'PAPER_CROP_SCALE',
        'PAPER_BACKBONES', 'PAPER_CLUSTER_NUMBER', 'PAPER_SEED', 'ATTACK_DEVICE', 'ATTENTION_DEVICE',
        'ATTENTION_MODE', 'NUM_PHASES', 'HOTSPOT_TOP_K', 'HOTSPOT_IOU', 'GUIDE_CAPTION_ROLE',
        'GUIDE_TEXT_OVERRIDE', 'GUIDE_OVERRIDES', 'EXTRA_CONCEPTS', 'EXTRA_CONCEPTS_BY_SAMPLE',
        'ATTENTION_MODEL_ID', 'ATTENTION_REVISION', 'ATTENTION_LAYER', 'ATTENTION_INPUT_RES',
        'MAX_CONCEPT_TOKENS', 'CONTENT_WORDS_ONLY', 'ATTENTION_PROMPT', 'PRUNE_TEXT_BRANCH',
        'FOA_AUTOCAST', 'GRADIENT_SCALE', 'OBSERVE_ALL_CONCEPTS_EVERY', 'ASYNC_OBSERVATION',
        'PLOT_FIRST_N_SAMPLES', 'SHOW_FINAL_PLOTS', 'LIVE_PREVIEW', 'CONTINUE_ON_ERROR', 'SAMPLE_START']
    return {name: globals()[name] for name in names}

def pack_artifacts(directory):
    directory = Path(directory)
    archive = directory.with_suffix('.zip')
    temporary = Path(str(archive) + '.tmp')
    with zipfile.ZipFile(temporary, 'w', zipfile.ZIP_DEFLATED, compresslevel=ZIP_COMPRESSION_LEVEL, allowZip64=True) as stream:
        for path in sorted(directory.rglob('*')):
            if path.is_file():
                stream.write(path, path.relative_to(directory).as_posix())
    temporary.replace(archive)
    return archive

def run_attention_experiment():
    global RUN_DIR
    validate_devices()
    # Default-device allocations in FOA stay on GPU 0; worker uses a thread-local GPU 1 context.
    torch.cuda.set_device(torch.device(ATTACK_DEVICE))
    pairs, caption_origin = prepare_pairs()
    fingerprints = {name: hashlib.sha256(source.encode()).hexdigest() for name, source in CODE_SNAPSHOTS.items()}
    identity = {'config': current_run_config(), 'caption_origin': caption_origin,
        'selected_pairs': [{k: p[k] for k in ('index', 'sample_id', 'source_path', 'target_path', 'guide', 'concepts')} for p in pairs],
        'implementation_sha256': fingerprints, 'source_notebook_sha256': V32_SOURCE_NOTEBOOK_SHA256,
        'foa_revision': FOA_SOURCE_REVISION, 'kmeans_revision': KMEANS_REVISION}
    signature = hashlib.sha256(json.dumps(identity, sort_keys=True, ensure_ascii=False).encode()).hexdigest()
    resuming = RESUME_RUN_DIR is not None
    RUN_DIR = Path(RESUME_RUN_DIR) if resuming else OUTPUT_ROOT / f'kltn_attack_v3_2_{ATTENTION_MODE}_{utc_stamp()}'
    if resuming:
        previous = json.loads((RUN_DIR / 'run_config.json').read_text(encoding='utf-8'))
        if previous.get('resume_signature') != signature:
            raise ValueError('Resume configuration/data/code differs. Restore config or start a new run.')
        records_by_id = recover_completed_records(RUN_DIR, {pair['sample_id'] for pair in pairs})
        for record in records_by_id.values():
            validate_completed_artifacts(RUN_DIR, record, STEPS)
    else:
        RUN_DIR.mkdir(parents=True, exist_ok=False)
        (RUN_DIR / 'images').mkdir()
        config = {**identity, **current_run_config(), 'resume_signature': signature,
            'notebook': 'kltn-attack-v3.2.ipynb', 'logic_audit_version': 1, 'checkpoint_steps': checkpoint_steps(STEPS),
            'checkpoint_interval': max(1, STEPS // 10), 'dataset_root': str(DATASET_ROOT),
            'loss_history': 'reset per sample', 'seed_policy': 'PAPER_SEED + original source index',
            'guide_barrier': 'fresh checkpoint guide map before next update', 'saved_png_attention_input': True,
            'reference_provenance': REFERENCE_PROVENANCE, 'packages': PACKAGES}
        write_json(RUN_DIR / 'run_config.json', config)
        (RUN_DIR / 'code').mkdir()
        for name, source in CODE_SNAPSHOTS.items():
            (RUN_DIR / 'code' / f'{name}.py').write_text(source, encoding='utf-8')
        freeze = subprocess.run([sys.executable, '-m', 'pip', 'freeze'], capture_output=True, text=True, check=True)
        (RUN_DIR / 'requirements-resolved.txt').write_text(freeze.stdout, encoding='utf-8')
        records_by_id = {}
    manifest_path = RUN_DIR / 'manifest.jsonl'
    write_jsonl_atomic(manifest_path, records_by_id.values())
    status = {'status': 'running', 'phase': 'loading', 'samples': len(records_by_id),
              'requested_samples': len(pairs), 'errors_this_session': 0, 'resumed': resuming,
              'current_sample_id': None, 'sample_position': None, 'current_step': 0, 'steps_per_sample': STEPS}
    write_json(RUN_DIR / 'status.json', status)
    started = time.perf_counter()
    timings, active_sample, failures = [], None, []
    try:
        todo = [pair for pair in pairs if pair['sample_id'] not in records_by_id]
        if todo:
            status['phase'] = 'loading_foa'; write_json(RUN_DIR / 'status.json', status)
            print(f'Loading FOA {PAPER_BACKBONES} on {ATTACK_DEVICE}; autocast={FOA_AUTOCAST}')
            extractor, models = ensure_foa_models()
            saved_metadata = (json.loads((RUN_DIR / 'model_metadata.json').read_text(encoding='utf-8'))
                              if resuming and (RUN_DIR / 'model_metadata.json').is_file() else None)
            saved_revision = saved_metadata['attention']['resolved_revision'] if saved_metadata else None
            status['phase'] = 'loading_attention'; write_json(RUN_DIR / 'status.json', status)
            print(f'Loading {ATTENTION_MODEL_ID} on {ATTENTION_DEVICE}; revision={saved_revision or ATTENTION_REVISION}')
            backend = ensure_attention_backend(resolved_revision=saved_revision)
            if saved_metadata:
                if saved_revision != backend.metadata.get('resolved_revision'):
                    raise ValueError('Attention model revision differs from the saved run.')
                for saved_model, model in zip(saved_metadata.get('foa', []), models):
                    if saved_model.get('resolved_revision') != getattr(model.model.config, '_commit_hash', None):
                        raise ValueError('FOA model revision differs from the saved run; start a new run.')
            write_json(RUN_DIR / 'model_metadata.json', {'attention': backend.metadata,
                'foa': [{'backbone': name, 'model_id': getattr(model.model.config, '_name_or_path', None),
                         'resolved_revision': getattr(model.model.config, '_commit_hash', None),
                         'weights_dtype': 'float32', 'forward_autocast': FOA_AUTOCAST}
                        for name, model in zip(PAPER_BACKBONES, models)], 'torch': torch.__version__,
                'cuda': torch.version.cuda, 'gpus': {device: torch.cuda.get_device_name(torch.device(device))
                                                   for device in (ATTACK_DEVICE, ATTENTION_DEVICE)}})
        loading_seconds = time.perf_counter() - started
        status['phase'] = 'attack'; write_json(RUN_DIR / 'status.json', status)
        transform = preprocess_image()
        with tqdm(total=len(pairs) * STEPS, initial=len(records_by_id) * STEPS, desc=f'FOA {ATTENTION_MODE} (processed updates)', mininterval=2) as progress_bar:
            for position, pair in enumerate(pairs):
                stem = pair['sample_id']
                if stem in records_by_id:
                    continue
                sample_started = time.perf_counter()
                active_sample = RUN_DIR / 'samples' / stem
                if active_sample.exists():
                    # Preserve every failed attempt. Check resolved boundaries before moving.
                    backup = RUN_DIR / 'failed_samples' / f'{stem}_{utc_stamp()}'
                    if not active_sample.resolve().is_relative_to(RUN_DIR.resolve()) or not backup.resolve().is_relative_to(RUN_DIR.resolve()):
                        raise ValueError('Invalid sample artifact path.')
                    backup.parent.mkdir(parents=True, exist_ok=True)
                    shutil.move(str(active_sample), str(backup))
                active_sample.mkdir(parents=True, exist_ok=False)
                write_json(active_sample / 'sample_config.json', {**pair, 'seed': PAPER_SEED + pair['index']})
                write_json(active_sample / 'status.json', {'status': 'running', 'sample_id': stem})
                observer = None
                progress_done = 0
                def report(phase, step, *, refresh=True):
                    status.update(phase=phase, current_sample_id=stem, sample_position=position + 1,
                                  current_step=step, samples=len(records_by_id), errors_this_session=len(failures),
                                  updated_at_utc=utc_stamp())
                    write_json(RUN_DIR / 'status.json', status)
                    progress_bar.set_postfix_str(
                        f'sample={position + 1}/{len(pairs)} id={stem} step={step}/{STEPS} '
                        f'ok={len(records_by_id)} errors={len(failures)} phase={phase}', refresh=refresh)
                report('preparing_sample', 0)
                print(f'Sample {position + 1}/{len(pairs)} | id={stem} | fixed guide={pair["guide"]!r}')
                try:
                    set_foa_seed(PAPER_SEED + pair['index'])
                    loss_fn = FOAOptimalTransportLoss(models, PAPER_CLUSTER_NUMBER)
                    with Image.open(pair['source_path']) as image:
                        source = transform(image).unsqueeze(0).to(ATTACK_DEVICE)
                    with Image.open(pair['target_path']) as image:
                        target = transform(image).unsqueeze(0).to(ATTACK_DEVICE)
                    paths = {role: f'images/{stem}_{role}.png' for role in ('source', 'target', 'adversarial')}
                    (RUN_DIR / paths['source']).write_bytes(tensor_png_bytes(source))
                    (RUN_DIR / paths['target']).write_bytes(tensor_png_bytes(target))
                    observer = CheckpointObserver(active_sample, pair['concepts'], backend, live=LIVE_PREVIEW, total_steps=STEPS, epsilon=PAPER_EPSILON)
                    def progress(step):
                        nonlocal progress_done
                        report('attack', step, refresh=False)
                        progress_bar.update(step - progress_done)
                        progress_done = step
                    adversarial, trace = foa_attention_attack(source, target, extractor, loss_fn,
                        observer=observer, sample_dir=active_sample, mode=ATTENTION_MODE,
                        on_step=progress, on_phase=report, steps=STEPS, epsilon=PAPER_EPSILON, alpha=PAPER_ALPHA)
                    report('saving_artifacts', STEPS)
                    expected = set(checkpoint_steps(STEPS))
                    if observer.saved_steps != expected or observer.finished_steps != expected:
                        raise AssertionError('Checkpoint/map set is incomplete.')
                    shutil.copy2(active_sample / 'checkpoints' / f'step_{STEPS:05d}.png', RUN_DIR / paths['adversarial'])
                    with Image.open(RUN_DIR / paths['source']) as img:
                        original = np.asarray(img).astype(np.int16)
                    with Image.open(RUN_DIR / paths['adversarial']) as img:
                        saved = np.asarray(img).astype(np.int16)
                    linf = int(np.abs(saved - original).max())
                    if linf > math.ceil(PAPER_EPSILON):
                        raise AssertionError('Final PNG exceeds perturbation budget.')
                    figures = []
                    if PLOT_FIRST_N_SAMPLES is None or position < PLOT_FIRST_N_SAMPLES:
                        figures = plot_attention_timeline(active_sample, show=SHOW_FINAL_PLOTS)
                    record = {'index': pair['index'], 'sample_id': stem, 'original_path': pair['source_path'],
                        'target_path': pair['target_path'], 'linf_255': linf, **paths,
                        'image_sha256': {role: sha256_file(RUN_DIR / path) for role, path in paths.items()},
                        'attention_mode': ATTENTION_MODE, 'attention_dir': f'samples/{stem}/attention',
                        'guide_concept': pair['guide'], 'checkpoints': checkpoint_steps(STEPS)}
                    timing = {'sample_id': stem, 'total_seconds': time.perf_counter() - sample_started,
                        'checkpoint_attention_seconds': observer.attention_seconds,
                        'guide_wait_seconds': observer.guide_wait_seconds, 'steps': STEPS,
                        'checkpoints': len(expected), 'seed': PAPER_SEED + pair['index']}
                    write_json(active_sample / 'timing.json', timing)
                    # Commit manifest before complete status, so interrupted commits are retried safely.
                    write_json(active_sample / 'manifest_record.json', record)
                    committed = {**records_by_id, stem: record}
                    write_jsonl_atomic(manifest_path, committed.values())
                    write_json(active_sample / 'status.json', {'status': 'complete', 'sample_id': stem,
                        'linf_255': linf, 'checkpoints': len(expected), 'figures': figures})
                    records_by_id[stem] = record
                    timings.append(timing)
                    report('sample_complete', STEPS)
                except Exception as error:
                    failure = {'sample_id': stem, 'status': 'error', 'error_type': type(error).__name__, 'error': str(error)}
                    failures.append(failure)
                    report('sample_error', progress_done)
                    write_json(active_sample / 'status.json', failure)
                    (active_sample / 'traceback.txt').write_text(traceback.format_exc(), encoding='utf-8')
                    with (RUN_DIR / 'errors.jsonl').open('a', encoding='utf-8') as stream:
                        stream.write(json.dumps({**failure, 'time_utc': utc_stamp()}, ensure_ascii=False) + '\n')
                    if not CONTINUE_ON_ERROR:
                        raise
                    print(f'Sample {stem} failed: {type(error).__name__}: {error}')
                finally:
                    if observer is not None:
                        observer.close()
                    source = target = adversarial = trace = loss_fn = observer = None
                    progress_bar.update(STEPS - progress_done)
                status.update(samples=len(records_by_id), errors_this_session=len(failures))
                write_json(RUN_DIR / 'status.json', status)
        status.update(status='complete' if len(records_by_id) == len(pairs) else 'partial', phase='complete',
                      current_sample_id=None, sample_position=None, current_step=None)
    except (Exception, KeyboardInterrupt) as error:
        status.update(status='interrupted' if isinstance(error, KeyboardInterrupt) else 'error', phase='stopped',
                      error_type=type(error).__name__, error=str(error))
        (RUN_DIR / 'traceback.txt').write_text(traceback.format_exc(), encoding='utf-8')
        if active_sample is not None and active_sample.name not in records_by_id:
            write_json(active_sample / 'status.json', {**status, 'sample_id': active_sample.name})
        print(f'Run stopped: {type(error).__name__}: {error}. Partial artifacts are retained.')
    finally:
        status.update(samples=len(records_by_id), errors_this_session=len(failures))
        write_json(RUN_DIR / 'status.json', status)
        total_seconds = time.perf_counter() - started
        write_json(RUN_DIR / 'summary.json', {'status': status['status'], 'completed_samples': len(records_by_id),
            'requested_samples': len(pairs), 'loading_seconds': locals().get('loading_seconds'),
            'session_seconds': total_seconds, 'per_sample_this_session': timings, 'failures_this_session': failures,
            'seconds_per_completed_sample_this_session': sum(t['total_seconds'] for t in timings) / len(timings) if timings else None})
    print('Output:', RUN_DIR, '| status:', status['status'])
    return RUN_DIR

remember_code('run')
if RUN_ATTACK:
    run_attention_experiment()
else:
    print('Definitions ready. RUN_ATTACK=True starts/resumes a run.')


## Xem lại một mẫu và đổi concept sau chạy

Cell inspect vẽ timeline từ maps có sẵn, không tải lại Qwen. Cell reprobe tính maps mới cho
tất cả PNG đã lưu, không chạy lại attack và không sửa checkpoint. Muốn mở run cũ trong session mới,
đặt `RUN_ATTACK=False`, chạy các cell định nghĩa, sửa đường dẫn ở cell cần dùng rồi chạy cell đó.
Chỉ reprobe: có thể dùng một GPU với `ATTENTION_DEVICE='cuda:0'`.

In [ ]:
RUN_INSPECT = False
INSPECT_RUN_DIR = None       # None: run hiện tại; hoặc Path('/kaggle/input/.../run_folder')
INSPECT_SAMPLE_ID = '00000'
if RUN_INSPECT:
    inspected_run = RUN_DIR if INSPECT_RUN_DIR is None else Path(INSPECT_RUN_DIR)
    if inspected_run is None:
        raise ValueError('Choose INSPECT_RUN_DIR.')
    plot_attention_timeline(inspected_run / 'samples' / INSPECT_SAMPLE_ID,
        figure_dir=OUTPUT_ROOT / 'inspections' / f'{INSPECT_SAMPLE_ID}_{utc_stamp()}', show=True)


In [ ]:
RUN_REPROBE = False
REPROBE_RUN_DIR = None      # vd Path('/kaggle/input/your-attack-run/kltn_v3_fixed_...')
REPROBE_SAMPLE_ID = '00000'    # None: mẫu đầu; hoặc '00000'
REPROBE_CONCEPTS = {'custom_0': 'a panda bear', 'custom_1': 'a dog with a toothbrush'}

def reprobe_saved_checkpoints(run_dir, sample_id, concepts, backend, analysis_root=None):
    run_dir = Path(run_dir)
    sample_dir = run_dir / 'samples' / sample_id
    files = sorted((sample_dir / 'checkpoints').glob('step_*.png'))
    if not files or int(files[0].stem.split('_')[-1]) != 0:
        raise ValueError('Saved run needs at least its step-0 PNG checkpoint.')
    if not concepts:
        raise ValueError('Provide at least one concept.')
    for label, text in concepts.items():
        if not isinstance(label, str) or not label or not all(c.isalnum() or c in '_-' for c in label):
            raise ValueError('Concept labels must be safe nonempty identifiers.')
        if not isinstance(text, str) or not text.strip():
            raise ValueError('Concept text cannot be empty.')
    # Default writable destination, including when run_dir is read-only Kaggle Input.
    root = OUTPUT_ROOT / 'analyses' if analysis_root is None else Path(analysis_root)
    destination = root / f'{sample_id}_{utc_stamp()}'
    destination.mkdir(parents=True, exist_ok=False)
    attention_root = destination / 'attention'
    attention_root.mkdir()
    write_json(attention_root / 'concepts.json', concepts)
    image_hashes = {str(path): sha256_file(path) for path in files}
    status = {'status': 'running', 'checkpoints': 0}
    write_json(destination / 'status.json', status)
    write_json(destination / 'run_config.json', {'source_run': str(run_dir), 'sample_id': sample_id,
               'concepts': concepts, 'checkpoint_sha256': image_hashes,
               'attention_model': backend.metadata, 'prompt': ATTENTION_PROMPT,
               'content_words_only': CONTENT_WORDS_ONLY, 'max_concept_tokens': MAX_CONCEPT_TOKENS})
    initial_box = None
    initial_maps = {}
    try:
        original_metrics = sample_dir / 'attention' / 'checkpoint_metrics.jsonl'
        if original_metrics.is_file():
            first = next(line for line in original_metrics.read_text(encoding='utf-8').splitlines() if line.strip())
            initial_box = json.loads(first)['initial_hotspot_box']
        with preserve_rng(), (attention_root / 'checkpoint_metrics.jsonl').open('w', encoding='utf-8') as metric_stream:
            with Image.open(files[0]) as image:
                original = np.asarray(image.convert('RGB')).astype(np.int16)
            for path in files:
                step = int(path.stem.split('_')[-1])
                output_dir = attention_root / f'step_{step:05d}'; output_dir.mkdir()
                with Image.open(path) as image:
                    image = image.convert('RGB')
                    results, by_text = {}, {}
                    for label, text in concepts.items():
                        if text not in by_text:
                            by_text[text] = backend.extract(image, text)
                        result = by_text[text]; results[label] = result
                        np.savez_compressed(output_dir / f'{label}.npz', probability=result['probability'], minmax=result['minmax'])
                        write_json(output_dir / f'{label}.json', result['metadata'])
                    if initial_box is None:
                        first_result = next(iter(results.values()))
                        initial_box = build_schedule(first_result['minmax'], NUM_PHASES, HOTSPOT_TOP_K, HOTSPOT_IOU)[0]['box']
                    metrics = {}
                    for label, result in results.items():
                        probability = result['probability']
                        initial_maps.setdefault(label, probability.copy())
                        metrics[label] = {'initial_hotspot_mass': hotspot_grid_mass(probability, initial_box),
                            'l1_vs_initial': float(np.abs(probability - initial_maps[label]).sum()),
                            'mean_raw_vision_mass': result['metadata']['mean_raw_vision_mass']}
                    record = {'step': step, 'initial_hotspot_box': initial_box, 'concept_metrics': metrics,
                              'linf_saved_255': int(np.abs(np.asarray(image).astype(np.int16) - original).max())}
                    metric_stream.write(json.dumps(record, allow_nan=False) + '\n'); metric_stream.flush()
                    status['checkpoints'] += 1
                    write_json(destination / 'status.json', status)
        plot_attention_timeline(sample_dir, attention_root, destination / 'figures', show=True)
        status['status'] = 'complete'
    except (Exception, KeyboardInterrupt) as error:
        status.update(status='interrupted' if isinstance(error, KeyboardInterrupt) else 'error',
                      error_type=type(error).__name__, error=str(error))
        (destination / 'traceback.txt').write_text(traceback.format_exc(), encoding='utf-8')
        raise
    finally:
        unchanged = all(sha256_file(path) == digest for path, digest in image_hashes.items())
        status['original_checkpoints_unchanged'] = unchanged
        write_json(destination / 'status.json', status)
        if not unchanged:
            raise AssertionError('Original checkpoint bytes changed during reprobe.')
    archive = pack_artifacts(destination)
    print('Reprobe output:', destination); display(FileLink(str(archive)))
    return destination

remember_code('reprobe')
if RUN_REPROBE:
    selected_run = RUN_DIR if REPROBE_RUN_DIR is None else Path(REPROBE_RUN_DIR)
    if selected_run is None:
        raise ValueError('Set REPROBE_RUN_DIR or run an attack first.')
    candidates = sorted(path.name for path in (selected_run / 'samples').iterdir() if path.is_dir())
    if not candidates:
        raise ValueError('No sample directories in selected run.')
    selected_id = candidates[0] if REPROBE_SAMPLE_ID is None else REPROBE_SAMPLE_ID
    reprobe_saved_checkpoints(selected_run, selected_id, REPROBE_CONCEPTS, ensure_attention_backend())
else:
    print('Reprobe is off. Set RUN_REPROBE=True to analyze saved checkpoints with new text.')


In [ ]:
# Can run separately after interruption/failure to export partial artifacts.
if RUN_DIR is not None and RUN_DIR.is_dir():
    archive = pack_artifacts(RUN_DIR)
    print('Run status:', json.loads((RUN_DIR / 'status.json').read_text(encoding='utf-8')))
    for path in (RUN_DIR / 'manifest.jsonl', RUN_DIR / 'run_config.json', RUN_DIR / 'summary.json', archive):
        if path.is_file():
            display(FileLink(str(path)))
else:
    print('No attack run to export in this session.')
